In [ ]:
from typing import List, Dict, Optional
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import json
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from scipy.stats import pearsonr 
import spacy
import pandas as pd
import re
from tqdm import tqdm
import os
from datetime import datetime

In [ ]:
import transformers
import sklearn
print(torch.__version__)
print(transformers.__version__)
print(sklearn.__version__)
print(spacy.__version__)

In [ ]:
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}") # For the first GPU
    print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
import pickle
from tqdm import tqdm
def save_pkl(tgt_list, svg_path):
    with open(svg_path, "wb") as f:
        pickle.dump(tgt_list, f)

def load_pkl(path) :
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

# 0. Dataset Creation

In [ ]:
from dataclasses import dataclass, field

# Examples commented at bottom of cell

@dataclass
class SRLFrame:
    predicate_word_idx: int
    labels: List[str]
    predicate_form: Optional[str] = None
    arg_head_idx: Optional[List[int]] = None

@dataclass
class UtteranceSample:
    words: List[str]
    frames: List[SRLFrame]
    politeness: Optional[float] = None

    
# def create_utterance_samples(data_path):
#     samples = []
#     with open(data_path, 'r', encoding='utf-8') as f:
#         for line in f:
#             data = json.loads(line)
#             samples.append(UtteranceSample(
#                 words=data["words"],
#                 frames=[SRLFrame(**fr) for fr in data["srl_frames"]],
#                 politeness=data.get("politeness", None)
#             ))
#     return samples

def create_utterance_samples(data_path):
    samples = []
    with open(data_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            samples.append(UtteranceSample(
                words=data["words"],
                frames=[SRLFrame(**fr) for fr in data["srl_frames"]],
                politeness=data.get("politeness", None)
            ))
    return samples


# 1. SRL Dataset Embedding Creation (Utterance level)

In [ ]:
# ----------------------------
# Dataset (NEW: utterance-level with multiple SRL frames)
# ----------------------------

class SRLDataset(Dataset):
    """
    NEW behavior:
      - One dataset item = one utterance (words) that may contain MULTIPLE SRL frames (predicates).
      - We build BERT inputs ONCE per utterance as:
          [CLS] <utterance (wordpiece)> [SEP]
        (**changed**: we do NOT append predicate tokens here, because there are multiple predicates.)

      - We keep:
          - wordpiece indices for each word's FIRST subtoken (to pool BERT to word level)
          - utterance length
          - per-frame:
              * predicate_word_idx
              * labels (BIO) -> label_ids
              * role_ids and arg masks
              * arg_head_idx (optional)
      - politeness is utterance-level.
    """
    def __init__(
        self,
        samples: List["UtteranceSample"],   # **changed**: SRLSample -> UtteranceSample
        tokenizer,
        label2id: Dict[str, int],
        max_length: int = 256,
        debug_print: bool = False
    ):
        self.samples = samples
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.id2label = {v: k for k, v in label2id.items()}
        self.max_length = max_length
        self.debug_print = debug_print

    def __len__(self):
        return len(self.samples)

    def _tokenize_utterance(self, words: List[str]):
        # Tokenize utterance as split words to preserve word boundaries
        enc = self.tokenizer(
            words,
            is_split_into_words=True,
            add_special_tokens=False,
            return_attention_mask=False,
            return_token_type_ids=False
        )
        return enc  # dict with 'input_ids'

#     def _build_word_first_wp_fullidx(self, words: List[str]) -> List[int]:
#         """
#         Returns list length n_words:
#             word_first_wp_fullidx[w] = position of first subtoken of word w
#         Positions are in the tokenization output that includes special tokens,
#         which we will align to [CLS] + sentence_wp + [SEP].
#         """
#         n_words = len(words)

#         tmp = self.tokenizer(words, is_split_into_words=True, return_offsets_mapping=False)
#         word_ids = tmp.word_ids()

#         first_pos_by_wid = {}
#         for pos, wid in enumerate(word_ids):
#             if wid is not None and wid not in first_pos_by_wid:
#                 first_pos_by_wid[wid] = pos

#         word_first_wp_fullidx = [first_pos_by_wid[w] for w in range(n_words)]
#         return word_first_wp_fullidx


    def _build_word_first_wp_fullidx(self, words):
        tmp = self.tokenizer(
            words,
            is_split_into_words=True,
            add_special_tokens=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            truncation=False,
        )

        first_pos_by_wid = {}
        for pos, wid in enumerate(tmp.word_ids()):
            if wid is not None and wid not in first_pos_by_wid:
                first_pos_by_wid[wid] = pos

        # +1 if you later prepend [CLS] manually
        return [first_pos_by_wid[w] + 1 if w in first_pos_by_wid else -1 for w in range(len(words))]

    def _normalize_srl_label(self, lbl: str) -> str:
        # **changed**: strip PropBank reference/continuation markers
        # Example: "I-R-ARGM-TMP" -> "I-ARGM-TMP"
        return lbl.replace("R-", "").replace("C-", "")

    
    def _frame_to_tensors(self, words: List[str], frame) -> Dict[str, torch.Tensor]:
        """
        Convert a single SRLFrame into tensor fields aligned to word-level length.
        """
        n_words = len(words)
        assert 0 <= frame.predicate_word_idx < n_words, "Bad predicate index."

        # Labels -> IDs
        # label_ids = [self.label2id[lbl] for lbl in frame.labels]
        # assert len(label_ids) == n_words
        
        raw_labels = frame.labels
        norm_labels = [self._normalize_srl_label(lbl) for lbl in raw_labels]

        # ---- 2) label ids with safe fallback ----
        unk_id = self.label2id.get("O", 0)
        label_ids = [self.label2id.get(lbl, unk_id) for lbl in norm_labels]
        assert len(label_ids) == n_words, f"Frame labels length {len(label_ids)} != n_words {n_words}"


        # role_ids & masks (same logic as before but per frame)
        role_ids = [0] * n_words
        for i, tag in enumerate(frame.labels):
            if i == frame.predicate_word_idx:
                role_ids[i] = 1
            if "ARG0" in tag:
                role_ids[i] = 2
            elif "ARG1" in tag:
                role_ids[i] = 3
            elif "ARG2" in tag:
                role_ids[i] = 4
            elif "ARGM" in tag:
                role_ids[i] = 5

        # arg0_mask = [1 if "ARG0" in tag else 0 for tag in frame.labels]
        # arg1_mask = [1 if "ARG1" in tag else 0 for tag in frame.labels]
        # arg2_mask = [1 if "ARG2" in tag else 0 for tag in frame.labels]
        # argm_mask = [1 if "ARGM" in tag else 0 for tag in frame.labels]
        
        arg0_mask = [1 if "ARG0" in tag else 0 for tag in norm_labels]
        arg1_mask = [1 if "ARG1" in tag else 0 for tag in norm_labels]
        arg2_mask = [1 if "ARG2" in tag else 0 for tag in norm_labels]
        argm_mask = [1 if "ARGM" in tag else 0 for tag in norm_labels]

        out = {
            "labels": torch.tensor(label_ids, dtype=torch.long),  # [n_words]
            "pred_word_idx": torch.tensor(frame.predicate_word_idx, dtype=torch.long),
            "role_ids": torch.tensor(role_ids, dtype=torch.long),
            "arg0_mask": torch.tensor(arg0_mask, dtype=torch.long),
            "arg1_mask": torch.tensor(arg1_mask, dtype=torch.long),
            "arg2_mask": torch.tensor(arg2_mask, dtype=torch.long),
            "argm_mask": torch.tensor(argm_mask, dtype=torch.long),
        }
        if getattr(frame, "arg_head_idx", None) is not None:
            out["arg_head_idx"] = torch.tensor(frame.arg_head_idx, dtype=torch.long)

        return out

    def __getitem__(self, idx):
        instance = self.samples[idx]
        words = instance.words
        n_words = len(words)

        # ----------------------------
        # Build utterance-level BERT inputs ONCE
        # ----------------------------
        enc_utt = self._tokenize_utterance(words)
        sent_wp_ids = enc_utt["input_ids"]

        # **changed**: utterance-only input (no predicate appended)
        input_ids = [self.tokenizer.cls_token_id] + sent_wp_ids + [self.tokenizer.sep_token_id]
        ttids = [0] * len(input_ids)  # single segment
        attention_mask = [1] * len(input_ids)

        # wordpiece mapping (word -> first subtoken position)
        word_first_wp_fullidx = self._build_word_first_wp_fullidx(words)

        # Truncate if needed
        if len(input_ids) > self.max_length:
            input_ids = input_ids[:self.max_length]
            ttids = ttids[:self.max_length]
            attention_mask = attention_mask[:self.max_length]
            max_pos = self.max_length - 1
            word_first_wp_fullidx = [min(p, max_pos) for p in word_first_wp_fullidx]

        if self.debug_print:
            toks_debug = self.tokenizer.convert_ids_to_tokens(input_ids, skip_special_tokens=False)
            print(f"[DeBug idx={idx}] " + " ".join(toks_debug))

        # ----------------------------
        # NEW: per-frame tensors (variable length list)
        # ----------------------------
        frame_dicts = []
        for fr in instance.frames:   # **changed**: iterate frames
            # safety: skip empty frames (if your data contains {})
            if fr is None:
                continue
            if getattr(fr, "labels", None) is None or getattr(fr, "predicate_word_idx", None) is None:
                continue
            frame_dicts.append(self._frame_to_tensors(words, fr))

        res = {
            # utterance-level tensors
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "token_type_ids": torch.tensor(ttids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "word_first_wp_fullidx": torch.tensor(word_first_wp_fullidx, dtype=torch.long),  # [n_words]
            "sent_len": torch.tensor(n_words, dtype=torch.long),

            # **changed**: frames is a LIST of dicts
            "frames": frame_dicts,
        }

        # utterance-level target
        if instance.politeness is not None:
            res["politeness"] = torch.tensor(instance.politeness, dtype=torch.float)

        return res


In [ ]:
def srl_collate(batch: List[Dict], pad_token_id: int, pad_label_id: int = -100):
    """
    Pads full BERT inputs to same length; also pads word-level tensors to max sentence length.
    Returns tensors ready for the model.
    """
    B = len(batch)
    max_L = max(item["input_ids"].size(0) for item in batch)
    # print("This is B {}, max_L {}".format(B,max_L))
    #make tensor B rows and Max_L columns
    input_ids = torch.full((B, max_L), pad_token_id, dtype=torch.long)
    token_type_ids = torch.zeros((B, max_L), dtype=torch.long)
    attention_mask = torch.zeros((B, max_L), dtype=torch.long)

    # Word-level padding
    max_n = max(int(item["sent_len"]) for item in batch)
    word_first_wp_fullidx = torch.full((B, max_n), -1, dtype=torch.long)
    labels = torch.full((B, max_n), pad_label_id, dtype=torch.long)
    sent_lens = torch.zeros((B,), dtype=torch.long)
    pred_word_idx = torch.zeros((B,), dtype=torch.long)
    sentence_mask = torch.zeros((B, max_n), dtype=torch.bool)
    role_ids = torch.zeros((B, max_n), dtype=torch.long)
    arg0_mask = torch.zeros((B, max_n), dtype=torch.long)
    arg1_mask = torch.zeros((B, max_n), dtype=torch.long)
    arg2_mask = torch.zeros((B, max_n), dtype=torch.long)
    argm_mask = torch.zeros((B, max_n), dtype=torch.long)
    
    has_politeness = "politeness" in batch[0]
    if has_politeness:
        politeness = torch.zeros((B,), dtype=torch.float)
    # TODO add other models

    for i, item in enumerate(batch):
        # print("This is item {}".format(item))
        L = item["input_ids"].size(0)
        input_ids[i, :L] = item["input_ids"]
        token_type_ids[i, :L] = item["token_type_ids"]
        attention_mask[i, :L] = item["attention_mask"]

        n = int(item["sent_len"])
        word_first_wp_fullidx[i, :n] = item["word_first_wp_fullidx"]
        labels[i, :n] = item["labels"]
        sent_lens[i] = n
        pred_word_idx[i] = item["pred_word_idx"]
        sentence_mask[i, :n] = True
        role_ids[i, :n] = item["role_ids"]
        arg0_mask[i, :n] = item["arg0_mask"]
        arg1_mask[i, :n] = item["arg1_mask"]
        arg2_mask[i, :n] = item["arg2_mask"]
        argm_mask[i, :n] = item["argm_mask"]
        
        if has_politeness:
            politeness[i] = item["politeness"]
        # TODO add for other models

    res = {
        "input_ids": input_ids,
        "token_type_ids": token_type_ids,
        "attention_mask": attention_mask,
        "word_first_wp_fullidx": word_first_wp_fullidx,  # [B, max_n] (full-seq positions; -1 for pad)
        "sentence_mask": sentence_mask,                  # [B, max_n] (bool mask for valid words)
        "labels": labels,                                # [B, max_n] (pad_label_id for pad)
        "sent_lens": sent_lens,                          # [B]
        "pred_word_idx": pred_word_idx,
        "role_ids": role_ids,
        "arg0_mask": arg0_mask,
        "arg1_mask": arg1_mask,
        "arg2_mask": arg2_mask,
        "argm_mask": argm_mask,
    }
    
    if has_politeness:
        res["politeness"] = politeness
        
    return res

# 2. Collate (Utterance level)

In [ ]:
import torch
from typing import List, Dict

def srl_collate_ulevel(batch: List[Dict], pad_token_id: int, pad_label_id: int = -100):
    """
    NEW behavior:
      - Pads BERT inputs to same wp length (max_L)
      - Pads word-level tensors to max word length (max_n)
      - **NEW**: Pads SRL frames to max number of frames (max_F) and
               pads per-frame word-level tensors to max_n.

    Output shapes (key ones):
      input_ids:           [B, max_L]
      word_first_wp_fullidx:[B, max_n]
      sentence_mask:       [B, max_n]

      frames_labels:       [B, max_F, max_n]
      frames_pred_word_idx:[B, max_F]
      frames_mask:         [B, max_F]  (True if frame exists)

      frames_role_ids:     [B, max_F, max_n]
      frames_arg0_mask:    [B, max_F, max_n]  ... etc
    """
    B = len(batch)

    # ----------------------------
    # 1) Pad BERT-level sequence
    # ----------------------------
    max_L = max(item["input_ids"].size(0) for item in batch)

    input_ids = torch.full((B, max_L), pad_token_id, dtype=torch.long)
    # **changed**: token_type_ids now exists but is always 0 segment in new dataset;
    # still pad/copy for completeness
    token_type_ids = torch.zeros((B, max_L), dtype=torch.long)
    attention_mask = torch.zeros((B, max_L), dtype=torch.long)

    # ----------------------------
    # 2) Pad word-level sequence
    # ----------------------------
    max_n = max(int(item["sent_len"]) for item in batch)

    word_first_wp_fullidx = torch.full((B, max_n), -1, dtype=torch.long)
    sent_lens = torch.zeros((B,), dtype=torch.long)
    sentence_mask = torch.zeros((B, max_n), dtype=torch.bool)

    # ----------------------------
    # 3) **NEW**: Pad frames (variable count per utterance)
    # ----------------------------
    # **changed**: labels/pred_word_idx/role_ids are per-frame now, not per-utterance
    max_F = max(len(item.get("frames", [])) for item in batch)

    frames_mask = torch.zeros((B, max_F), dtype=torch.bool)  # **new**

    frames_labels = torch.full((B, max_F, max_n), pad_label_id, dtype=torch.long)  # **new**
    frames_pred_word_idx = torch.zeros((B, max_F), dtype=torch.long)              # **new**

    frames_role_ids = torch.zeros((B, max_F, max_n), dtype=torch.long)            # **new**
    frames_arg0_mask = torch.zeros((B, max_F, max_n), dtype=torch.long)           # **new**
    frames_arg1_mask = torch.zeros((B, max_F, max_n), dtype=torch.long)           # **new**
    frames_arg2_mask = torch.zeros((B, max_F, max_n), dtype=torch.long)           # **new**
    frames_argm_mask = torch.zeros((B, max_F, max_n), dtype=torch.long)           # **new**

    # Optional: arg_head_idx (if present in any frame)
    has_arg_head = any(
        any(("arg_head_idx" in fr) for fr in item.get("frames", []))
        for item in batch
    )
    if has_arg_head:
        # we don't know arg_head_idx length (often small like [pred, arg0_head,...]),
        # so we pad to max length across all frames in batch
        max_H = 0
        for item in batch:
            for fr in item.get("frames", []):
                if "arg_head_idx" in fr:
                    max_H = max(max_H, int(fr["arg_head_idx"].numel()))
        frames_arg_head_idx = torch.full((B, max_F, max_H), -1, dtype=torch.long)  # **new**

    # ----------------------------
    # 4) Utterance-level targets
    # ----------------------------
    has_politeness = "politeness" in batch[0]
    if has_politeness:
        politeness = torch.zeros((B,), dtype=torch.float)

    # ----------------------------
    # Fill tensors
    # ----------------------------
    for i, item in enumerate(batch):
        # ---- BERT-level ----
        L = item["input_ids"].size(0)
        input_ids[i, :L] = item["input_ids"]
        token_type_ids[i, :L] = item["token_type_ids"]
        attention_mask[i, :L] = item["attention_mask"]

        # ---- word-level ----
        n = int(item["sent_len"])
        word_first_wp_fullidx[i, :n] = item["word_first_wp_fullidx"]
        sent_lens[i] = n
        sentence_mask[i, :n] = True

        # ---- utterance target ----
        if has_politeness:
            politeness[i] = item["politeness"]

        # ---- frames ----
        frames = item.get("frames", [])
        for f, fr in enumerate(frames):
            frames_mask[i, f] = True  # mark this frame as valid

            # per-frame word-level fields (pad to max_n)
            frames_labels[i, f, :n] = fr["labels"]
            frames_pred_word_idx[i, f] = fr["pred_word_idx"]

            frames_role_ids[i, f, :n] = fr["role_ids"]
            frames_arg0_mask[i, f, :n] = fr["arg0_mask"]
            frames_arg1_mask[i, f, :n] = fr["arg1_mask"]
            frames_arg2_mask[i, f, :n] = fr["arg2_mask"]
            frames_argm_mask[i, f, :n] = fr["argm_mask"]

            if has_arg_head and ("arg_head_idx" in fr):
                h = int(fr["arg_head_idx"].numel())
                frames_arg_head_idx[i, f, :h] = fr["arg_head_idx"]

    # ----------------------------
    # Package output
    # ----------------------------
    res = {
        "input_ids": input_ids,
        "token_type_ids": token_type_ids,
        "attention_mask": attention_mask,

        "word_first_wp_fullidx": word_first_wp_fullidx,  # [B, max_n]
        "sentence_mask": sentence_mask,                  # [B, max_n]
        "sent_lens": sent_lens,                          # [B]

        # **changed**: SRL supervision is now per-frame
        "frames_mask": frames_mask,                      # **new** [B, max_F]
        "frames_labels": frames_labels,                  # **new** [B, max_F, max_n]
        "frames_pred_word_idx": frames_pred_word_idx,    # **new** [B, max_F]

        "frames_role_ids": frames_role_ids,              # **new** [B, max_F, max_n]
        "frames_arg0_mask": frames_arg0_mask,            # **new** [B, max_F, max_n]
        "frames_arg1_mask": frames_arg1_mask,            # **new** [B, max_F, max_n]
        "frames_arg2_mask": frames_arg2_mask,            # **new** [B, max_F, max_n]
        "frames_argm_mask": frames_argm_mask,            # **new** [B, max_F, max_n]
    }

    if has_politeness:
        res["politeness"] = politeness

    if has_arg_head:
        res["frames_arg_head_idx"] = frames_arg_head_idx  # **new** [B, max_F, max_H]

    return res


## 4. SRL Models

## DirectionalSRL (ARG0 --> ARG2 + ARGM) - Utterance Level

### Without fusion + mlp

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class DirectionalSRL_nofusion_mlp(nn.Module):
    """
    SRL-aware BERT model for politeness prediction with directional SRL only
    (no gated fusion; last-layer concatenation only).

    Directional design:
      - Query primarily from ARG0, fallback to predicate
      - Attend to ARG2 + ARGM if present, otherwise full sentence

    Comparison setting:
      - Keeps the same directional SRL pipeline as the fusion model
      - Keeps last-layer concatenation: [query_source ; context]
      - Adds the extra MLP head for a cleaner apple-to-apple comparison
      - Uses bert-base-uncased through the bert_name argument
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        use_indicator: bool = True,
        indicator_dim: int = 10,
        lstm_hidden: int = 768,
        mlp_hidden: int = 300,   # kept only for compatibility; not used
        dropout: float = 0.1,
        use_distance: bool = True,
        pos_dim: int = 50,
        max_distance: int = 128,
        num_roles: int = 6,
        role_dim: int = 32,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)

        self.use_indicator = use_indicator
        self.use_distance = use_distance
        self.max_distance = max_distance

        bert_dim = self.config.hidden_size
        in_dim = bert_dim + (indicator_dim if use_indicator else 0)

        # ---------------------------------------------------------
        # role indicator embeddings
        # ---------------------------------------------------------
        if use_indicator:
            self.indicator_emb = nn.Embedding(num_roles, indicator_dim)

        # ---------------------------------------------------------
        # relative distance embeddings from predicate
        # ---------------------------------------------------------
        if use_distance:
            self.pos_emb = nn.Embedding(2 * max_distance + 1, pos_dim)
            in_dim += pos_dim

        # ---------------------------------------------------------
        # BiLSTM over word-level sequence
        # ---------------------------------------------------------
        self.bilstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=lstm_hidden // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)

        # ---------------------------------------------------------
        # directional attention projections
        # ---------------------------------------------------------
        self.W_q = nn.Linear(lstm_hidden, attn_dim)
        self.W_k = nn.Linear(lstm_hidden, attn_dim)
        self.W_v = nn.Linear(lstm_hidden, attn_dim)
        self.attn_layer_norm = nn.LayerNorm(attn_dim)

        # ---------------------------------------------------------
        # CHANGED: extra MLP head
        # Now:
        #   Linear(lstm_hidden + attn_dim, mlp_hidden)
        #   -> ReLU -> Dropout -> Linear(mlp_hidden, 1)
        #
        self.politeness_head = nn.Sequential(
            nn.Linear(lstm_hidden + attn_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

        self.global_only_head = nn.Sequential(
            nn.Linear(bert_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

    def masked_mean(self, x, mask):
        # x: [B, N, D], mask: [B, N]
        mask = mask.float()
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        return (x * mask.unsqueeze(-1)).sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        token_type_ids,
        attention_mask,
        word_first_wp_fullidx,
        sentence_mask,
        sent_lens,

        # per-frame inputs
        frames_pred_word_idx=None,     # [B, F]
        frames_role_ids=None,          # [B, F, N]
        frames_arg0_mask=None,         # [B, F, N]
        frames_arg2_mask=None,         # [B, F, N]
        frames_argm_mask=None,         # [B, F, N]
        frames_mask=None,              # [B, F]

        politeness=None,
        **kwargs
    ):
        """
        Output:
          - score_utt: [B] utterance-level politeness score
          - loss: scalar
          - attn_weights: [B, F, N]
        """
        device = input_ids.device
        B, L = input_ids.size()
        N = sentence_mask.size(1)

        # ------------------------------------------------------------
        # 1) BERT once per utterance -> word-level embeddings once
        # ------------------------------------------------------------
        bert_out = self.bert(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
        )
        H = bert_out.last_hidden_state  # [B, L, D]

        gather_idx = word_first_wp_fullidx.clone()
        gather_idx[gather_idx < 0] = 0
        gather_idx = gather_idx.unsqueeze(-1).expand(-1, -1, H.size(-1))
        H_words = torch.gather(H, dim=1, index=gather_idx)                # [B, N, D]
        H_words = H_words * sentence_mask.unsqueeze(-1)                   # mask pads

        # ------------------------------------------------------------
        # fallback if no frame info
        # CHANGED: use simple linear fallback, no MLP
        # ------------------------------------------------------------
        if frames_mask is None or frames_pred_word_idx is None:
            pooled = self.masked_mean(H_words, sentence_mask)
            score = self.global_only_head(self.dropout(pooled)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if politeness is not None:
                loss = nn.MSELoss()(score, politeness)
            attn_weights = torch.zeros((B, 1, N), device=device)
            return score, loss, attn_weights

        # ------------------------------------------------------------
        # 2) Flatten frames: treat (B,F) as batch dimension
        # ------------------------------------------------------------
        Bf = frames_mask.size(0)
        F = frames_mask.size(1)
        assert Bf == B, "Batch size mismatch."

        BF = B * F

        Hf = H_words.unsqueeze(1).expand(B, F, N, H_words.size(-1)).contiguous()
        Hf = Hf.view(BF, N, H_words.size(-1))                                     # [BF,N,D]

        sent_mask_f = sentence_mask.unsqueeze(1).expand(B, F, N).contiguous().view(BF, N)
        lengths_f = sent_lens.unsqueeze(1).expand(B, F).contiguous().view(BF)
        pred_idx_f = frames_pred_word_idx.contiguous().view(BF)

        X = Hf

        # ------------------------------------------------------------
        # indicator features per frame
        # ------------------------------------------------------------
        if self.use_indicator and frames_role_ids is not None:
            role_ids_f = frames_role_ids.clamp(0, 5).contiguous().view(BF, N)
            X = torch.cat([X, self.indicator_emb(role_ids_f)], dim=-1)

        # ------------------------------------------------------------
        # distance features per frame
        # ------------------------------------------------------------
        if self.use_distance:
            positions = torch.arange(N, device=device).unsqueeze(0).expand(BF, -1)
            rel = (positions - pred_idx_f.unsqueeze(1)).clamp(-self.max_distance, self.max_distance)
            rel = rel + self.max_distance
            X = torch.cat([X, self.pos_emb(rel)], dim=-1)

        # ------------------------------------------------------------
        # 3) BiLSTM per valid frame
        # ------------------------------------------------------------
        valid_frame_mask = frames_mask.reshape(-1).bool()

        X_valid = X[valid_frame_mask]
        lengths_valid = lengths_f[valid_frame_mask].clamp(min=1)

        if valid_frame_mask.sum() == 0:
            pooled = self.masked_mean(H_words, sentence_mask)
            score = self.global_only_head(self.dropout(pooled)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if politeness is not None:
                loss = nn.MSELoss()(score, politeness)
            attn_weights = torch.zeros((B, 1, N), device=device)
            return score, loss, attn_weights

        packed = pack_padded_sequence(
            X_valid,
            lengths=lengths_valid.detach().cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        G_packed, _ = self.bilstm(packed)
        G_valid, _ = pad_packed_sequence(G_packed, batch_first=True, total_length=N)
        G_valid = self.dropout(G_valid)

        H_lstm = G_valid.size(-1)
        G = torch.zeros((X.size(0), N, H_lstm), device=G_valid.device, dtype=G_valid.dtype)
        G[valid_frame_mask] = G_valid

        # ------------------------------------------------------------
        # 4) ARG0 -> (ARG2 + ARGM) attention per frame
        # ------------------------------------------------------------
        arg0_f = frames_arg0_mask.contiguous().view(BF, N) if frames_arg0_mask is not None else None
        arg2_f = frames_arg2_mask.contiguous().view(BF, N) if frames_arg2_mask is not None else None
        argm_f = frames_argm_mask.contiguous().view(BF, N) if frames_argm_mask is not None else None

        batch_idx = torch.arange(BF, device=device)
        gp = G[batch_idx, pred_idx_f.clamp(min=0, max=N - 1), :]  # [BF,H]

        # Query source = mean(ARG0); fallback to predicate
        if arg0_f is not None:
            arg0_sum = arg0_f.sum(dim=1, keepdim=True)
            has_arg0 = (arg0_sum > 0).float()
            denom = arg0_sum.clamp(min=1.0)
            arg0_vec = (G * arg0_f.unsqueeze(-1)).sum(dim=1) / denom
            query_source = has_arg0 * arg0_vec + (1 - has_arg0) * gp
        else:
            query_source = gp

        # Target mask = (ARG2 + ARGM) ∩ sentence_mask
        if arg2_f is not None and argm_f is not None:
            target_mask = (arg2_f + argm_f).clamp(max=1)
            target_mask = target_mask * sent_mask_f
        else:
            target_mask = sent_mask_f

        Q = self.W_q(query_source).unsqueeze(1)   # [BF,1,A]
        K = self.W_k(G)                           # [BF,N,A]
        V = self.W_v(G)                           # [BF,N,A]

        attn_scores = torch.bmm(Q, K.transpose(1, 2)) / (K.size(-1) ** 0.5)
        attn_scores = attn_scores.masked_fill(target_mask.unsqueeze(1) == 0, -10000.0)

        attn_weights = torch.softmax(attn_scores, dim=-1)  # [BF,1,N]
        context = torch.bmm(attn_weights, V).squeeze(1)    # [BF,A]
        context = self.attn_layer_norm(context)

        # ------------------------------------------------------------
        # CHANGED: keep last-layer concatenation, but no extra MLP
        # features = [query_source ; context]
        # ------------------------------------------------------------
        features = torch.cat([query_source, context], dim=1)  # [BF, H+A]

        # CHANGED: unbounded regression output, no 4*sigmoid
        score_frame = self.politeness_head(self.dropout(features)).squeeze(-1)  # [BF]

        # ------------------------------------------------------------
        # 5) Aggregate frames -> utterance score using frames_mask
        # ------------------------------------------------------------
        score_frame = score_frame.view(B, F)                       # [B,F]
        attn_weights = attn_weights.squeeze(1).view(B, F, N)       # [B,F,N]

        fm = frames_mask.float()
        denom = fm.sum(dim=1).clamp(min=1.0)
        score_utt = (score_frame * fm).sum(dim=1) / denom

        attn_weights = attn_weights * fm.unsqueeze(-1)

        loss = torch.tensor(0.0, device=device)
        if politeness is not None:
            loss = nn.MSELoss()(score_utt, politeness)

        return score_utt, loss, attn_weights

### Without fusion + no mlp

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class DirectionalSRL_nofusion_nomlp(nn.Module):
    """
    SRL-aware BERT model for politeness prediction with directional SRL only
    (no gated fusion; last-layer concatenation only).

    Directional design:
      - Query primarily from ARG0, fallback to predicate
      - Attend to ARG2 + ARGM if present, otherwise full sentence

    Comparison setting:
      - Keeps the same directional SRL pipeline as the fusion model
      - Keeps last-layer concatenation: [query_source ; context]
      - Removes the extra MLP head for a cleaner apple-to-apple comparison
      - Uses bert-base-uncased through the bert_name argument
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        use_indicator: bool = True,
        indicator_dim: int = 10,
        lstm_hidden: int = 768,
        mlp_hidden: int = 300,   # kept only for compatibility; not used
        dropout: float = 0.1,
        use_distance: bool = True,
        pos_dim: int = 50,
        max_distance: int = 128,
        num_roles: int = 6,
        role_dim: int = 32,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)

        self.use_indicator = use_indicator
        self.use_distance = use_distance
        self.max_distance = max_distance

        bert_dim = self.config.hidden_size
        in_dim = bert_dim + (indicator_dim if use_indicator else 0)

        # ---------------------------------------------------------
        # role indicator embeddings
        # ---------------------------------------------------------
        if use_indicator:
            self.indicator_emb = nn.Embedding(num_roles, indicator_dim)

        # ---------------------------------------------------------
        # relative distance embeddings from predicate
        # ---------------------------------------------------------
        if use_distance:
            self.pos_emb = nn.Embedding(2 * max_distance + 1, pos_dim)
            in_dim += pos_dim

        # ---------------------------------------------------------
        # BiLSTM over word-level sequence
        # ---------------------------------------------------------
        self.bilstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=lstm_hidden // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)

        # ---------------------------------------------------------
        # directional attention projections
        # ---------------------------------------------------------
        self.W_q = nn.Linear(lstm_hidden, attn_dim)
        self.W_k = nn.Linear(lstm_hidden, attn_dim)
        self.W_v = nn.Linear(lstm_hidden, attn_dim)
        self.attn_layer_norm = nn.LayerNorm(attn_dim)

        # ---------------------------------------------------------
        # CHANGED: remove extra MLP head
        # before:
        #   Linear(lstm_hidden + attn_dim, mlp_hidden)
        #   -> ReLU -> Dropout -> Linear(mlp_hidden, 1)
        #
        # now:
        #   Linear(lstm_hidden + attn_dim, 1)
        # ---------------------------------------------------------
        self.politeness_head = nn.Linear(lstm_hidden + attn_dim, 1)

        # ---------------------------------------------------------
        # OPTIONAL fallback head if no frame info exists
        # also simplified for consistency
        # ---------------------------------------------------------
        self.global_only_head = nn.Linear(bert_dim, 1)

    def masked_mean(self, x, mask):
        # x: [B, N, D], mask: [B, N]
        mask = mask.float()
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        return (x * mask.unsqueeze(-1)).sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        token_type_ids,
        attention_mask,
        word_first_wp_fullidx,
        sentence_mask,
        sent_lens,

        # per-frame inputs
        frames_pred_word_idx=None,     # [B, F]
        frames_role_ids=None,          # [B, F, N]
        frames_arg0_mask=None,         # [B, F, N]
        frames_arg2_mask=None,         # [B, F, N]
        frames_argm_mask=None,         # [B, F, N]
        frames_mask=None,              # [B, F]

        politeness=None,
        **kwargs
    ):
        """
        Output:
          - score_utt: [B] utterance-level politeness score
          - loss: scalar
          - attn_weights: [B, F, N]
        """
        device = input_ids.device
        B, L = input_ids.size()
        N = sentence_mask.size(1)

        # ------------------------------------------------------------
        # 1) BERT once per utterance -> word-level embeddings once
        # ------------------------------------------------------------
        bert_out = self.bert(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
        )
        H = bert_out.last_hidden_state  # [B, L, D]

        gather_idx = word_first_wp_fullidx.clone()
        gather_idx[gather_idx < 0] = 0
        gather_idx = gather_idx.unsqueeze(-1).expand(-1, -1, H.size(-1))
        H_words = torch.gather(H, dim=1, index=gather_idx)                # [B, N, D]
        H_words = H_words * sentence_mask.unsqueeze(-1)                   # mask pads

        # ------------------------------------------------------------
        # fallback if no frame info
        # CHANGED: use simple linear fallback, no MLP
        # ------------------------------------------------------------
        if frames_mask is None or frames_pred_word_idx is None:
            pooled = self.masked_mean(H_words, sentence_mask)
            score = self.global_only_head(self.dropout(pooled)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if politeness is not None:
                loss = nn.MSELoss()(score, politeness)
            attn_weights = torch.zeros((B, 1, N), device=device)
            return score, loss, attn_weights

        # ------------------------------------------------------------
        # 2) Flatten frames: treat (B,F) as batch dimension
        # ------------------------------------------------------------
        Bf = frames_mask.size(0)
        F = frames_mask.size(1)
        assert Bf == B, "Batch size mismatch."

        BF = B * F

        Hf = H_words.unsqueeze(1).expand(B, F, N, H_words.size(-1)).contiguous()
        Hf = Hf.view(BF, N, H_words.size(-1))                                     # [BF,N,D]

        sent_mask_f = sentence_mask.unsqueeze(1).expand(B, F, N).contiguous().view(BF, N)
        lengths_f = sent_lens.unsqueeze(1).expand(B, F).contiguous().view(BF)
        pred_idx_f = frames_pred_word_idx.contiguous().view(BF)

        X = Hf

        # ------------------------------------------------------------
        # indicator features per frame
        # ------------------------------------------------------------
        if self.use_indicator and frames_role_ids is not None:
            role_ids_f = frames_role_ids.clamp(0, 5).contiguous().view(BF, N)
            X = torch.cat([X, self.indicator_emb(role_ids_f)], dim=-1)

        # ------------------------------------------------------------
        # distance features per frame
        # ------------------------------------------------------------
        if self.use_distance:
            positions = torch.arange(N, device=device).unsqueeze(0).expand(BF, -1)
            rel = (positions - pred_idx_f.unsqueeze(1)).clamp(-self.max_distance, self.max_distance)
            rel = rel + self.max_distance
            X = torch.cat([X, self.pos_emb(rel)], dim=-1)

        # ------------------------------------------------------------
        # 3) BiLSTM per valid frame
        # ------------------------------------------------------------
        valid_frame_mask = frames_mask.reshape(-1).bool()

        X_valid = X[valid_frame_mask]
        lengths_valid = lengths_f[valid_frame_mask].clamp(min=1)

        if valid_frame_mask.sum() == 0:
            pooled = self.masked_mean(H_words, sentence_mask)
            score = self.global_only_head(self.dropout(pooled)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if politeness is not None:
                loss = nn.MSELoss()(score, politeness)
            attn_weights = torch.zeros((B, 1, N), device=device)
            return score, loss, attn_weights

        packed = pack_padded_sequence(
            X_valid,
            lengths=lengths_valid.detach().cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        G_packed, _ = self.bilstm(packed)
        G_valid, _ = pad_packed_sequence(G_packed, batch_first=True, total_length=N)
        G_valid = self.dropout(G_valid)

        H_lstm = G_valid.size(-1)
        G = torch.zeros((X.size(0), N, H_lstm), device=G_valid.device, dtype=G_valid.dtype)
        G[valid_frame_mask] = G_valid

        # ------------------------------------------------------------
        # 4) ARG0 -> (ARG2 + ARGM) attention per frame
        # ------------------------------------------------------------
        arg0_f = frames_arg0_mask.contiguous().view(BF, N) if frames_arg0_mask is not None else None
        arg2_f = frames_arg2_mask.contiguous().view(BF, N) if frames_arg2_mask is not None else None
        argm_f = frames_argm_mask.contiguous().view(BF, N) if frames_argm_mask is not None else None

        batch_idx = torch.arange(BF, device=device)
        gp = G[batch_idx, pred_idx_f.clamp(min=0, max=N - 1), :]  # [BF,H]

        # Query source = mean(ARG0); fallback to predicate
        if arg0_f is not None:
            arg0_sum = arg0_f.sum(dim=1, keepdim=True)
            has_arg0 = (arg0_sum > 0).float()
            denom = arg0_sum.clamp(min=1.0)
            arg0_vec = (G * arg0_f.unsqueeze(-1)).sum(dim=1) / denom
            query_source = has_arg0 * arg0_vec + (1 - has_arg0) * gp
        else:
            query_source = gp

        # Target mask = (ARG2 + ARGM) ∩ sentence_mask
        if arg2_f is not None and argm_f is not None:
            target_mask = (arg2_f + argm_f).clamp(max=1)
            target_mask = target_mask * sent_mask_f
        else:
            target_mask = sent_mask_f

        Q = self.W_q(query_source).unsqueeze(1)   # [BF,1,A]
        K = self.W_k(G)                           # [BF,N,A]
        V = self.W_v(G)                           # [BF,N,A]

        attn_scores = torch.bmm(Q, K.transpose(1, 2)) / (K.size(-1) ** 0.5)
        attn_scores = attn_scores.masked_fill(target_mask.unsqueeze(1) == 0, -10000.0)

        attn_weights = torch.softmax(attn_scores, dim=-1)  # [BF,1,N]
        context = torch.bmm(attn_weights, V).squeeze(1)    # [BF,A]
        context = self.attn_layer_norm(context)

        # ------------------------------------------------------------
        # CHANGED: keep last-layer concatenation, but no extra MLP
        # features = [query_source ; context]
        # ------------------------------------------------------------
        features = torch.cat([query_source, context], dim=1)  # [BF, H+A]

        # CHANGED: unbounded regression output, no 4*sigmoid
        score_frame = self.politeness_head(self.dropout(features)).squeeze(-1)  # [BF]

        # ------------------------------------------------------------
        # 5) Aggregate frames -> utterance score using frames_mask
        # ------------------------------------------------------------
        score_frame = score_frame.view(B, F)                       # [B,F]
        attn_weights = attn_weights.squeeze(1).view(B, F, N)       # [B,F,N]

        fm = frames_mask.float()
        denom = fm.sum(dim=1).clamp(min=1.0)
        score_utt = (score_frame * fm).sum(dim=1) / denom

        attn_weights = attn_weights * fm.unsqueeze(-1)

        loss = torch.tensor(0.0, device=device)
        if politeness is not None:
            loss = nn.MSELoss()(score_utt, politeness)

        return score_utt, loss, attn_weights

### With fusion and MLP layer (additional)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

#_Fusion_ARG0_ARG2_ARGM
class DirectionalSRL_fusion_mlp(nn.Module):
    """
    SRL-aware BERT model for politeness prediction with directional SRL + gated fusion.

    Directional design:
      - Query primarily from ARG0, fallback to predicate
      - Attend to ARG2 + ARGM if present, otherwise full sentence

    Fusion design:
      - Build a directional SRL representation from [query_source ; context]
      - Build a global text representation from the utterance
      - Use gated residual fusion so SRL selectively updates the global text representation
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        use_indicator: bool = True,
        indicator_dim: int = 10,
        lstm_hidden: int = 768,
        mlp_hidden: int = 300,
        dropout: float = 0.1,
        use_distance: bool = True,
        pos_dim: int = 50,
        max_distance: int = 128,
        num_roles: int = 6,
        role_dim: int = 32,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)

        self.use_indicator = use_indicator
        self.use_distance = use_distance
        self.max_distance = max_distance

        bert_dim = self.config.hidden_size
        in_dim = bert_dim + (indicator_dim if use_indicator else 0)

        # ---------------------------------------------------------
        # # role indicator embeddings
        # ---------------------------------------------------------
        if use_indicator:
            self.indicator_emb = nn.Embedding(num_roles, indicator_dim)

        # ---------------------------------------------------------
        # # relative distance embeddings from predicate
        # ---------------------------------------------------------
        if use_distance:
            self.pos_emb = nn.Embedding(2 * max_distance + 1, pos_dim)
            in_dim += pos_dim

        # ---------------------------------------------------------
        # # BiLSTM over word-level sequence
        # ---------------------------------------------------------
        self.bilstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=lstm_hidden // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)

        # ---------------------------------------------------------
        # # directional attention
        # ---------------------------------------------------------
        self.W_q = nn.Linear(lstm_hidden, attn_dim)
        self.W_k = nn.Linear(lstm_hidden, attn_dim)
        self.W_v = nn.Linear(lstm_hidden, attn_dim)
        self.attn_layer_norm = nn.LayerNorm(attn_dim)

        # ---------------------------------------------------------
        # # NEW: global utterance branch
        # ---------------------------------------------------------
        self.global_proj = nn.Linear(bert_dim, attn_dim)
        self.global_to_hidden = nn.Linear(attn_dim, lstm_hidden)

        # ---------------------------------------------------------
        # # NEW: structural branch projection
        # # SRL structure = [query_source ; context]
        # ---------------------------------------------------------
        self.struct_proj = nn.Sequential(
            nn.Linear(lstm_hidden + attn_dim, lstm_hidden),
            nn.LayerNorm(lstm_hidden),
            nn.Tanh()
        )

        # ---------------------------------------------------------
        # # NEW: fusion gate
        # # gate from [query_source ; context ; global_f]
        # ---------------------------------------------------------
        self.fusion_gate = nn.Linear(lstm_hidden + attn_dim + attn_dim, lstm_hidden)

        # ---------------------------------------------------------
        # # NEW: final politeness head now uses fused representation
        # ---------------------------------------------------------
        self.politeness_head = nn.Sequential(
            nn.Linear(lstm_hidden, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

        # ---------------------------------------------------------
        # # NEW: fallback head if no frame info exists
        # ---------------------------------------------------------
        self.global_only_head = nn.Sequential(
            nn.Linear(bert_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

    def masked_mean(self, x, mask):
        # x: [B, N, D], mask: [B, N]
        mask = mask.float()
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        return (x * mask.unsqueeze(-1)).sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        token_type_ids,
        attention_mask,
        word_first_wp_fullidx,
        sentence_mask,
        sent_lens,
        frames_pred_word_idx=None,     # [B, F]
        frames_role_ids=None,          # [B, F, N]
        frames_arg0_mask=None,         # [B, F, N]
        frames_arg2_mask=None,         # [B, F, N]
        frames_argm_mask=None,         # [B, F, N]
        frames_mask=None,              # [B, F]
        politeness=None,
        **kwargs
    ):
        """
        Output:
          - score_utt: [B] utterance-level politeness score
          - loss: scalar
          - attn_weights: [B, F, N]
        """
        device = input_ids.device
        B, _ = input_ids.size()
        N = sentence_mask.size(1)

        # ---------------------------------------------------------
        # # 1) BERT once per utterance -> word-level embeddings once
        # ---------------------------------------------------------
        bert_out = self.bert(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
        )
        H = bert_out.last_hidden_state  # [B, L, D]

        gather_idx = word_first_wp_fullidx.clone()
        gather_idx[gather_idx < 0] = 0
        gather_idx = gather_idx.unsqueeze(-1).expand(-1, -1, H.size(-1))
        H_words = torch.gather(H, dim=1, index=gather_idx)          # [B, N, D]
        H_words = H_words * sentence_mask.unsqueeze(-1)             # mask pads

        # ---------------------------------------------------------
        # # NEW: global utterance vector
        # ---------------------------------------------------------
        global_vec = self.masked_mean(H_words, sentence_mask)       # [B, D]

        # ---------------------------------------------------------
        # # fallback path if no frame information exists
        # ---------------------------------------------------------
        if frames_mask is None or frames_pred_word_idx is None:
            score = self.global_only_head(self.dropout(global_vec)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if politeness is not None:
                loss = nn.MSELoss()(score, politeness)
            return score, loss, torch.zeros((B, 1, N), device=device)

        # ---------------------------------------------------------
        # # 2) flatten frames: treat (B,F) as batch dimension
        # ---------------------------------------------------------
        F = frames_mask.size(1)
        BF = B * F

        Hf = H_words.unsqueeze(1).expand(B, F, N, H_words.size(-1)).contiguous()
        Hf = Hf.view(BF, N, H_words.size(-1))                                      # [BF,N,D]

        sent_mask_f = sentence_mask.unsqueeze(1).expand(B, F, N).contiguous().view(BF, N)
        lengths_f = sent_lens.unsqueeze(1).expand(B, F).contiguous().view(BF)
        pred_idx_f = frames_pred_word_idx.contiguous().view(BF)

        X = Hf

        # ---------------------------------------------------------
        # # add role indicator embeddings
        # ---------------------------------------------------------
        if self.use_indicator and frames_role_ids is not None:
            role_ids_f = frames_role_ids.clamp(0, 5).contiguous().view(BF, N)
            X = torch.cat([X, self.indicator_emb(role_ids_f)], dim=-1)

        # ---------------------------------------------------------
        # # add predicate-relative distance embeddings
        # ---------------------------------------------------------
        if self.use_distance:
            positions = torch.arange(N, device=device).unsqueeze(0).expand(BF, -1)
            rel = (positions - pred_idx_f.unsqueeze(1)).clamp(-self.max_distance, self.max_distance)
            rel = rel + self.max_distance
            X = torch.cat([X, self.pos_emb(rel)], dim=-1)

        # ---------------------------------------------------------
        # # 3) BiLSTM per valid frame
        # ---------------------------------------------------------
        valid_frame_mask = frames_mask.reshape(-1).bool()

        if valid_frame_mask.sum() == 0:
            score = self.global_only_head(self.dropout(global_vec)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if politeness is not None:
                loss = nn.MSELoss()(score, politeness)
            return score, loss, torch.zeros((B, 1, N), device=device)

        X_valid = X[valid_frame_mask]
        lengths_valid = lengths_f[valid_frame_mask].clamp(min=1)

        packed = pack_padded_sequence(
            X_valid,
            lengths=lengths_valid.detach().cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        G_packed, _ = self.bilstm(packed)
        G_valid, _ = pad_packed_sequence(G_packed, batch_first=True, total_length=N)
        G_valid = self.dropout(G_valid)

        H_lstm = G_valid.size(-1)
        G = torch.zeros((BF, N, H_lstm), device=device, dtype=G_valid.dtype)
        G[valid_frame_mask] = G_valid

        # ---------------------------------------------------------
        # # 4) ARG0 -> (ARG2 + ARGM) attention per frame
        # ---------------------------------------------------------
        arg0_f = frames_arg0_mask.contiguous().view(BF, N) if frames_arg0_mask is not None else None
        arg2_f = frames_arg2_mask.contiguous().view(BF, N) if frames_arg2_mask is not None else None
        argm_f = frames_argm_mask.contiguous().view(BF, N) if frames_argm_mask is not None else None

        batch_idx = torch.arange(BF, device=device)
        gp = G[batch_idx, pred_idx_f.clamp(min=0, max=N - 1), :]  # [BF,H]

        # ---------------------------------------------------------
        # # query from ARG0 mean, fallback to predicate
        # ---------------------------------------------------------
        if arg0_f is not None:
            arg0_sum = arg0_f.sum(dim=1, keepdim=True)
            has_arg0 = (arg0_sum > 0).float()
            arg0_vec = (G * arg0_f.unsqueeze(-1)).sum(dim=1) / arg0_sum.clamp(min=1.0)
            query_source = has_arg0 * arg0_vec + (1.0 - has_arg0) * gp
        else:
            query_source = gp

        # ---------------------------------------------------------
        # # target mask = ARG2 + ARGM, fallback to full sentence
        # ---------------------------------------------------------
        if arg2_f is not None and argm_f is not None:
            target_mask = (arg2_f + argm_f).clamp(max=1)
            target_mask = target_mask * sent_mask_f
        else:
            target_mask = sent_mask_f

        # ---------------------------------------------------------
        # # directional attention
        # ---------------------------------------------------------
        Q = self.W_q(query_source).unsqueeze(1)      # [BF,1,A]
        K = self.W_k(G)                              # [BF,N,A]
        V = self.W_v(G)                              # [BF,N,A]

        attn_scores = torch.bmm(Q, K.transpose(1, 2)) / (K.size(-1) ** 0.5)
        attn_scores = attn_scores.masked_fill(target_mask.unsqueeze(1) == 0, -10000.0)

        attn_weights = torch.softmax(attn_scores, dim=-1)  # [BF,1,N]
        context = torch.bmm(attn_weights, V).squeeze(1)    # [BF,A]
        context = self.attn_layer_norm(context)

        # ---------------------------------------------------------
        # # NEW: global text branch expanded to frames
        # ---------------------------------------------------------
        global_f = global_vec.unsqueeze(1).expand(B, F, global_vec.size(-1)).contiguous().view(BF, global_vec.size(-1))
        global_f = self.global_proj(global_f)              # [BF, A]
        global_h = self.global_to_hidden(self.dropout(global_f))  # [BF, H]

        # ---------------------------------------------------------
        # # NEW: structural SRL branch
        # ---------------------------------------------------------
        struct_input = torch.cat([query_source, context], dim=1)   # [BF, H+A]
        struct_h = self.struct_proj(self.dropout(struct_input))    # [BF, H]

        # ---------------------------------------------------------
        # # NEW: gated residual fusion
        # ---------------------------------------------------------
        gate_input = torch.cat([query_source, context, global_f], dim=1)   # [BF, H+A+A]
        fusion_gate = torch.sigmoid(self.fusion_gate(self.dropout(gate_input)))  # [BF, H]

        fused_frame = global_h + fusion_gate * struct_h   # [BF, H]

        # ---------------------------------------------------------
        # # CHANGED: politeness score from fused representation
        # ---------------------------------------------------------
        score_frame = self.politeness_head(self.dropout(fused_frame)).squeeze(-1)  # [BF]

        # ---------------------------------------------------------
        # # 5) aggregate frames -> utterance score using frames_mask
        # ---------------------------------------------------------
        score_frame = score_frame.view(B, F)                       # [B,F]
        attn_weights = attn_weights.squeeze(1).view(B, F, N)       # [B,F,N]

        fm = frames_mask.float()
        denom = fm.sum(dim=1).clamp(min=1.0)
        score_utt = (score_frame * fm).sum(dim=1) / denom

        attn_weights = attn_weights * fm.unsqueeze(-1)

        loss = torch.tensor(0.0, device=device)
        if politeness is not None:
            loss = nn.MSELoss()(score_utt, politeness)

        return score_utt, loss, attn_weights

### With fusion no additional MLP layer

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# _Fusion_ARG0_ARG2_ARGM_no MLP layer
class DirectionalSRL_fusion_nomlp(nn.Module):
    """
    SRL-aware BERT model for politeness prediction with directional SRL + gated fusion.

    Directional design:
      - Query primarily from ARG0, fallback to predicate
      - Attend to ARG2 + ARGM if present, otherwise full sentence

    Fusion design:
      - Build a directional SRL representation from [query_source ; context]
      - Build a global text representation from the utterance
      - Use gated residual fusion so SRL selectively updates the global text representation
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        use_indicator: bool = True,
        indicator_dim: int = 10,
        lstm_hidden: int = 768,
        mlp_hidden: int = 300,   # # kept for compatibility, but no longer used
        dropout: float = 0.1,
        use_distance: bool = True,
        pos_dim: int = 50,
        max_distance: int = 128,
        num_roles: int = 6,
        role_dim: int = 32,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)

        self.use_indicator = use_indicator
        self.use_distance = use_distance
        self.max_distance = max_distance

        bert_dim = self.config.hidden_size
        in_dim = bert_dim + (indicator_dim if use_indicator else 0)

        # ---------------------------------------------------------
        # # role indicator embeddings
        # ---------------------------------------------------------
        if use_indicator:
            self.indicator_emb = nn.Embedding(num_roles, indicator_dim)

        # ---------------------------------------------------------
        # # relative distance embeddings from predicate
        # ---------------------------------------------------------
        if use_distance:
            self.pos_emb = nn.Embedding(2 * max_distance + 1, pos_dim)
            in_dim += pos_dim

        # ---------------------------------------------------------
        # # BiLSTM over word-level sequence
        # ---------------------------------------------------------
        self.bilstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=lstm_hidden // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)

        # ---------------------------------------------------------
        # # directional attention
        # ---------------------------------------------------------
        self.W_q = nn.Linear(lstm_hidden, attn_dim)
        self.W_k = nn.Linear(lstm_hidden, attn_dim)
        self.W_v = nn.Linear(lstm_hidden, attn_dim)
        self.attn_layer_norm = nn.LayerNorm(attn_dim)

        # ---------------------------------------------------------
        # # global utterance branch
        # ---------------------------------------------------------
        self.global_proj = nn.Linear(bert_dim, attn_dim)
        self.global_to_hidden = nn.Linear(attn_dim, lstm_hidden)

        # ---------------------------------------------------------
        # # structural branch projection
        # # SRL structure = [query_source ; context]
        # ---------------------------------------------------------
        self.struct_proj = nn.Sequential(
            nn.Linear(lstm_hidden + attn_dim, lstm_hidden),
            nn.LayerNorm(lstm_hidden),
            nn.Tanh()
        )

        # ---------------------------------------------------------
        # # fusion gate
        # # gate from [query_source ; context ; global_f]
        # ---------------------------------------------------------
        self.fusion_gate = nn.Linear(lstm_hidden + attn_dim + attn_dim, lstm_hidden)

        # ---------------------------------------------------------
        # # CHANGED: remove extra MLP from politeness head
        # # before:
        # #   Linear(lstm_hidden, mlp_hidden) -> ReLU -> Dropout -> Linear(mlp_hidden, 1)
        # # now:
        # #   Linear(lstm_hidden, 1)
        # ---------------------------------------------------------
        self.politeness_head = nn.Linear(lstm_hidden, 1)

        # ---------------------------------------------------------
        # # CHANGED: remove extra MLP from fallback head
        # # before:
        # #   Linear(bert_dim, mlp_hidden) -> ReLU -> Dropout -> Linear(mlp_hidden, 1)
        # # now:
        # #   Linear(bert_dim, 1)
        # ---------------------------------------------------------
        self.global_only_head = nn.Linear(bert_dim, 1)

    def masked_mean(self, x, mask):
        # x: [B, N, D], mask: [B, N]
        mask = mask.float()
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        return (x * mask.unsqueeze(-1)).sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        token_type_ids,
        attention_mask,
        word_first_wp_fullidx,
        sentence_mask,
        sent_lens,
        frames_pred_word_idx=None,     # [B, F]
        frames_role_ids=None,          # [B, F, N]
        frames_arg0_mask=None,         # [B, F, N]
        frames_arg2_mask=None,         # [B, F, N]
        frames_argm_mask=None,         # [B, F, N]
        frames_mask=None,              # [B, F]
        politeness=None,
        **kwargs
    ):
        """
        Output:
          - score_utt: [B] utterance-level politeness score
          - loss: scalar
          - attn_weights: [B, F, N]
        """
        device = input_ids.device
        B, _ = input_ids.size()
        N = sentence_mask.size(1)

        # ---------------------------------------------------------
        # # 1) BERT once per utterance -> word-level embeddings once
        # ---------------------------------------------------------
        bert_out = self.bert(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
        )
        H = bert_out.last_hidden_state  # [B, L, D]

        gather_idx = word_first_wp_fullidx.clone()
        gather_idx[gather_idx < 0] = 0
        gather_idx = gather_idx.unsqueeze(-1).expand(-1, -1, H.size(-1))
        H_words = torch.gather(H, dim=1, index=gather_idx)          # [B, N, D]
        H_words = H_words * sentence_mask.unsqueeze(-1)             # mask pads

        # ---------------------------------------------------------
        # # global utterance vector
        # ---------------------------------------------------------
        global_vec = self.masked_mean(H_words, sentence_mask)       # [B, D]

        # ---------------------------------------------------------
        # # fallback path if no frame information exists
        # ---------------------------------------------------------
        if frames_mask is None or frames_pred_word_idx is None:
            score = self.global_only_head(self.dropout(global_vec)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if politeness is not None:
                loss = nn.MSELoss()(score, politeness)
            return score, loss, torch.zeros((B, 1, N), device=device)

        # ---------------------------------------------------------
        # # 2) flatten frames: treat (B,F) as batch dimension
        # ---------------------------------------------------------
        F = frames_mask.size(1)
        BF = B * F

        Hf = H_words.unsqueeze(1).expand(B, F, N, H_words.size(-1)).contiguous()
        Hf = Hf.view(BF, N, H_words.size(-1))                                      # [BF,N,D]

        sent_mask_f = sentence_mask.unsqueeze(1).expand(B, F, N).contiguous().view(BF, N)
        lengths_f = sent_lens.unsqueeze(1).expand(B, F).contiguous().view(BF)
        pred_idx_f = frames_pred_word_idx.contiguous().view(BF)

        X = Hf

        # ---------------------------------------------------------
        # # add role indicator embeddings
        # ---------------------------------------------------------
        if self.use_indicator and frames_role_ids is not None:
            role_ids_f = frames_role_ids.clamp(0, 5).contiguous().view(BF, N)
            X = torch.cat([X, self.indicator_emb(role_ids_f)], dim=-1)

        # ---------------------------------------------------------
        # # add predicate-relative distance embeddings
        # ---------------------------------------------------------
        if self.use_distance:
            positions = torch.arange(N, device=device).unsqueeze(0).expand(BF, -1)
            rel = (positions - pred_idx_f.unsqueeze(1)).clamp(-self.max_distance, self.max_distance)
            rel = rel + self.max_distance
            X = torch.cat([X, self.pos_emb(rel)], dim=-1)

        # ---------------------------------------------------------
        # # 3) BiLSTM per valid frame
        # ---------------------------------------------------------
        valid_frame_mask = frames_mask.reshape(-1).bool()

        if valid_frame_mask.sum() == 0:
            score = self.global_only_head(self.dropout(global_vec)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if politeness is not None:
                loss = nn.MSELoss()(score, politeness)
            return score, loss, torch.zeros((B, 1, N), device=device)

        X_valid = X[valid_frame_mask]
        lengths_valid = lengths_f[valid_frame_mask].clamp(min=1)

        packed = pack_padded_sequence(
            X_valid,
            lengths=lengths_valid.detach().cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        G_packed, _ = self.bilstm(packed)
        G_valid, _ = pad_packed_sequence(G_packed, batch_first=True, total_length=N)
        G_valid = self.dropout(G_valid)

        H_lstm = G_valid.size(-1)
        G = torch.zeros((BF, N, H_lstm), device=device, dtype=G_valid.dtype)
        G[valid_frame_mask] = G_valid

        # ---------------------------------------------------------
        # # 4) ARG0 -> (ARG2 + ARGM) attention per frame
        # ---------------------------------------------------------
        arg0_f = frames_arg0_mask.contiguous().view(BF, N) if frames_arg0_mask is not None else None
        arg2_f = frames_arg2_mask.contiguous().view(BF, N) if frames_arg2_mask is not None else None
        argm_f = frames_argm_mask.contiguous().view(BF, N) if frames_argm_mask is not None else None

        batch_idx = torch.arange(BF, device=device)
        gp = G[batch_idx, pred_idx_f.clamp(min=0, max=N - 1), :]  # [BF,H]

        # ---------------------------------------------------------
        # # query from ARG0 mean, fallback to predicate
        # ---------------------------------------------------------
        if arg0_f is not None:
            arg0_sum = arg0_f.sum(dim=1, keepdim=True)
            has_arg0 = (arg0_sum > 0).float()
            arg0_vec = (G * arg0_f.unsqueeze(-1)).sum(dim=1) / arg0_sum.clamp(min=1.0)
            query_source = has_arg0 * arg0_vec + (1.0 - has_arg0) * gp
        else:
            query_source = gp

        # ---------------------------------------------------------
        # # target mask = ARG2 + ARGM, fallback to full sentence
        # ---------------------------------------------------------
        if arg2_f is not None and argm_f is not None:
            target_mask = (arg2_f + argm_f).clamp(max=1)
            target_mask = target_mask * sent_mask_f
        else:
            target_mask = sent_mask_f

        # ---------------------------------------------------------
        # # directional attention
        # ---------------------------------------------------------
        Q = self.W_q(query_source).unsqueeze(1)      # [BF,1,A]
        K = self.W_k(G)                              # [BF,N,A]
        V = self.W_v(G)                              # [BF,N,A]

        attn_scores = torch.bmm(Q, K.transpose(1, 2)) / (K.size(-1) ** 0.5)
        attn_scores = attn_scores.masked_fill(target_mask.unsqueeze(1) == 0, -10000.0)

        attn_weights = torch.softmax(attn_scores, dim=-1)  # [BF,1,N]
        context = torch.bmm(attn_weights, V).squeeze(1)    # [BF,A]
        context = self.attn_layer_norm(context)

        # ---------------------------------------------------------
        # # global text branch expanded to frames
        # ---------------------------------------------------------
        global_f = global_vec.unsqueeze(1).expand(B, F, global_vec.size(-1)).contiguous().view(BF, global_vec.size(-1))
        global_f = self.global_proj(global_f)              # [BF, A]
        global_h = self.global_to_hidden(self.dropout(global_f))  # [BF, H]

        # ---------------------------------------------------------
        # # structural SRL branch
        # ---------------------------------------------------------
        struct_input = torch.cat([query_source, context], dim=1)   # [BF, H+A]
        struct_h = self.struct_proj(self.dropout(struct_input))    # [BF, H]

        # ---------------------------------------------------------
        # # gated residual fusion
        # ---------------------------------------------------------
        gate_input = torch.cat([query_source, context, global_f], dim=1)   # [BF, H+A+A]
        fusion_gate = torch.sigmoid(self.fusion_gate(self.dropout(gate_input)))  # [BF, H]

        fused_frame = global_h + fusion_gate * struct_h   # [BF, H]

        # ---------------------------------------------------------
        # # CHANGED: politeness score from fused representation
        # # no extra MLP here
        # ---------------------------------------------------------
        score_frame = self.politeness_head(self.dropout(fused_frame)).squeeze(-1)  # [BF]

        # ---------------------------------------------------------
        # # 5) aggregate frames -> utterance score using frames_mask
        # ---------------------------------------------------------
        score_frame = score_frame.view(B, F)                       # [B,F]
        attn_weights = attn_weights.squeeze(1).view(B, F, N)       # [B,F,N]

        fm = frames_mask.float()
        denom = fm.sum(dim=1).clamp(min=1.0)
        score_utt = (score_frame * fm).sum(dim=1) / denom

        attn_weights = attn_weights * fm.unsqueeze(-1)

        loss = torch.tensor(0.0, device=device)
        if politeness is not None:
            loss = nn.MSELoss()(score_utt, politeness)

        return score_utt, loss, attn_weights

# 5. Training Loop - Utterance level

In [ ]:
from scipy.stats import pearsonr
import torch
import torch.nn as nn


@torch.no_grad()
def eval_politeness(model, dataloader, device="cuda"):
    model.eval()
    total_loss, n_batches = 0.0, 0
    all_preds, all_golds = [], []

    for batch in dataloader:
        # **changed**: keep this simple because collate now returns only tensors
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

        preds, loss, _ = model(**batch)  # model returns (preds, loss, attn)

        total_loss += float(loss.item())
        n_batches += 1

        if "politeness" in batch:
            all_preds.append(preds.detach().cpu())
            all_golds.append(batch["politeness"].detach().cpu())

    avg_loss = total_loss / max(1, n_batches)

    if len(all_preds) > 0:
        all_preds = torch.cat(all_preds).numpy()
        all_golds = torch.cat(all_golds).numpy()
        corr = pearsonr(all_golds, all_preds)[0] if len(all_preds) > 1 else 0.0
        if corr != corr:  # NaN check
            corr = 0.0
    else:
        corr = 0.0

    return avg_loss, corr


def train_one_epoch(
    model,
    dataloader,
    optimizer,
    device="cuda",
    scheduler=None,
    grad_accum_steps=1,
    amp=True,
    max_grad_norm=1.0,
):
    model.train()
    total_loss, n_steps = 0.0, 0

    use_amp = amp and torch.cuda.is_available()
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(dataloader, 1):
        # **changed**: still works because batch is tensor-only with new collate
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

        with torch.amp.autocast('cuda', enabled=use_amp, dtype=torch.float16):
            _, loss, _ = model(**batch)

        total_loss += float(loss.detach().item())
        n_steps += 1

        loss = loss / grad_accum_steps

        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if step % grad_accum_steps == 0:
            if use_amp:
                scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            if use_amp:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)

            if scheduler is not None:
                scheduler.step()

    return total_loss / max(1, n_steps)


In [ ]:
mlp_layer = 300

### Training - with Fusion MLP layer

In [ ]:
import os
import json
import pickle
from datetime import datetime
from typing import List, Dict

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup


# ============================================================
# Data processing
# ============================================================

def data_processing_for_loader(
    train_dev: List[UtteranceSample],
    train_sample: List[UtteranceSample],
    dev_sample: List[UtteranceSample],
    test_sample: List[UtteranceSample],
    tokenizer,
):
    """
    Build label2id from all frame labels in train+dev.
    Return utterance-level SRLDataset objects.
    """
    label2id: Dict[str, int] = {}

    for s in train_dev:
        for fr in s.frames:
            if fr is None or fr.labels is None:
                continue
            for l in fr.labels:
                if l not in label2id:
                    label2id[l] = len(label2id)

    id2label = {v: k for k, v in label2id.items()}

    train_ds = SRLDataset(train_sample, tokenizer, label2id, max_length=128, debug_print=False)
    dev_ds   = SRLDataset(dev_sample,   tokenizer, label2id, max_length=128, debug_print=False)
    test_ds  = SRLDataset(test_sample,  tokenizer, label2id, max_length=128, debug_print=False)

    return train_ds, dev_ds, test_ds, label2id, id2label


# ============================================================
# Main training
# ============================================================
if __name__ == "__main__":
    # --------------------------------------------------------
    # 0) Setup
    # --------------------------------------------------------
    bert_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    base_path = "/Enter-your-path/SRL_processed/"
    train_file = base_path + "srl_politeness_train.aligned.jsonl"
    dev_file   = base_path + "srl_politeness_dev.aligned.jsonl"
    test_file  = base_path + "test_update/srl_politeness_test.aligned.jsonl"

    data_class_train = create_utterance_samples(train_file)
    data_class_dev   = create_utterance_samples(dev_file)
    data_class_test  = create_utterance_samples(test_file)

    train_dev_data = data_class_train + data_class_dev

    train_ds, dev_ds, test_ds, label2id, id2label = data_processing_for_loader(
        train_dev_data,
        data_class_train,
        data_class_dev,
        data_class_test,
        tokenizer
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate = lambda b: srl_collate_ulevel(b, pad_token_id=pad_token_id, pad_label_id=-100)

    train_loader = DataLoader(
        train_ds,
        batch_size=16,
        shuffle=True,
        collate_fn=collate,
        num_workers=0,   # easier debugging
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=16,
        shuffle=False,
        collate_fn=collate,
        num_workers=0,
    )

    # --------------------------------------------------------
    # 1) Build model (fusion directional, mlp_hidden=300)
    # --------------------------------------------------------
    model = DirectionalSRL_fusion_mlp(
        bert_name=bert_name,
        num_labels=len(label2id),
        use_indicator=True,
        use_distance=True,
        indicator_dim=10,
        lstm_hidden=768,
        mlp_hidden=mlp_layer,     # fixed as requested
        pos_dim=50,
        max_distance=128,
        dropout=0.1,
    ).to(device)

    # --------------------------------------------------------
    # 2) Optimizer / scheduler
    # --------------------------------------------------------
    num_epochs = 5
    grad_accum_steps = 1
    lr = 5e-5

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    total_steps = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # --------------------------------------------------------
    # 3) Save paths
    # --------------------------------------------------------
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")
    model_name = "directional_fusion_mlp"

    save_dir = "/Enter-your-path/model_log"
    hist_dir = "/Enter-your-path/loss_history"
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(hist_dir, exist_ok=True)

    best_path = os.path.join(save_dir, today + "_best_model_"+model_name+"_epoch_{}.ckpt".format(str(num_epochs)))
    hist_file_name = os.path.join(hist_dir, today + "_model_"+model_name+"_epoch_{}.pkl".format(str(num_epochs)))

    # --------------------------------------------------------
    # 4) Train loop
    # --------------------------------------------------------
    history = {"epoch": [], "train_loss": [], "dev_loss": [], "dev_corr": []}
    best_corr = -1.0

    print(f"Training for {num_epochs} epochs...")
    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device=device,
            scheduler=scheduler,
            grad_accum_steps=grad_accum_steps,
            amp=True,
        )

        dev_loss, dev_corr = eval_politeness(model, dev_loader, device=device)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_corr"].append(dev_corr)

        print(
            f"Epoch {epoch + 1}: "
            f"train_loss={tr_loss:.4f}  "
            f"dev_loss={dev_loss:.4f}  "
            f"dev_corr={dev_corr:.4f}"
        )

        if dev_corr > best_corr:
            best_corr = dev_corr
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "label2id": label2id,
                    "best_corr": best_corr,
                    "hparams": {
                        "architecture": "DirectionalSRL_fusion_mlp",
                        "bert_name": bert_name,
                        "num_labels": len(label2id),
                        "use_indicator": True,
                        "use_distance": True,
                        "indicator_dim": 10,
                        "lstm_hidden": 768,
                        "mlp_hidden": 300,
                        "pos_dim": 50,
                        "max_distance": 128,
                        "dropout": 0.1,
                        "batch_size": 16,
                        "lr": lr,
                        "num_epochs": num_epochs,
                        "max_length": 128,
                    },
                },
                best_path,
            )
            print("new best correlation saved")

    # --------------------------------------------------------
    # 5) Save history
    # --------------------------------------------------------
    with open(hist_file_name, "wb") as f:
        pickle.dump(history, f)

    print(f"Best model saved to {best_path}")
    print(f"History saved to {hist_file_name}")

### Training- with Fusion no MLP layer

In [ ]:
import os
import json
import pickle
from datetime import datetime
from typing import List, Dict

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup


# ============================================================
# Data processing
# ============================================================

def data_processing_for_loader(
    train_dev: List[UtteranceSample],
    train_sample: List[UtteranceSample],
    dev_sample: List[UtteranceSample],
    test_sample: List[UtteranceSample],
    tokenizer,
):
    """
    Build label2id from all frame labels in train+dev.
    Return utterance-level SRLDataset objects.
    """
    label2id: Dict[str, int] = {}

    for s in train_dev:
        for fr in s.frames:
            if fr is None or fr.labels is None:
                continue
            for l in fr.labels:
                if l not in label2id:
                    label2id[l] = len(label2id)

    id2label = {v: k for k, v in label2id.items()}

    train_ds = SRLDataset(train_sample, tokenizer, label2id, max_length=128, debug_print=False)
    dev_ds   = SRLDataset(dev_sample,   tokenizer, label2id, max_length=128, debug_print=False)
    test_ds  = SRLDataset(test_sample,  tokenizer, label2id, max_length=128, debug_print=False)

    return train_ds, dev_ds, test_ds, label2id, id2label


# ============================================================
# Main training
# ============================================================
if __name__ == "__main__":
    # --------------------------------------------------------
    # 0) Setup
    # --------------------------------------------------------
    bert_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    base_path = "/Enter-your-path/SRL_processed/"
    train_file = base_path + "srl_politeness_train.aligned.jsonl"
    dev_file   = base_path + "srl_politeness_dev.aligned.jsonl"
    test_file  = base_path + "test_update/srl_politeness_test.aligned.jsonl"

    data_class_train = create_utterance_samples(train_file)
    data_class_dev   = create_utterance_samples(dev_file)
    data_class_test  = create_utterance_samples(test_file)

    train_dev_data = data_class_train + data_class_dev

    train_ds, dev_ds, test_ds, label2id, id2label = data_processing_for_loader(
        train_dev_data,
        data_class_train,
        data_class_dev,
        data_class_test,
        tokenizer
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate = lambda b: srl_collate_ulevel(b, pad_token_id=pad_token_id, pad_label_id=-100)

    train_loader = DataLoader(
        train_ds,
        batch_size=16,
        shuffle=True,
        collate_fn=collate,
        num_workers=0,   # easier debugging
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=16,
        shuffle=False,
        collate_fn=collate,
        num_workers=0,
    )

    # --------------------------------------------------------
    # 1) Build model (fusion directional, mlp_hidden=256)
    # --------------------------------------------------------
    model = DirectionalSRL_fusion_nomlp(
        bert_name=bert_name,
        num_labels=len(label2id),
        use_indicator=True,
        use_distance=True,
        indicator_dim=10,
        lstm_hidden=768,
        mlp_hidden=mlp_layer,     # fixed as requested
        pos_dim=50,
        max_distance=128,
        dropout=0.1,
    ).to(device)

    # --------------------------------------------------------
    # 2) Optimizer / scheduler
    # --------------------------------------------------------
    num_epochs = 5
    grad_accum_steps = 1
    lr = 5e-5

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    total_steps = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # --------------------------------------------------------
    # 3) Save paths
    # --------------------------------------------------------
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")

    save_dir = "/Enter-your-path/model_log"
    hist_dir = "/Enter-your-path/loss_history"
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(hist_dir, exist_ok=True)

    best_path = os.path.join(save_dir, today + "_best_model_4_directional_fusion_no_mlp_epoch_{}.ckpt".format(str(num_epochs)))
    hist_file_name = os.path.join(hist_dir, today + "_model_directional_fusion_no_mlp_epoch_{}.pkl".format(str(num_epochs)))

    # --------------------------------------------------------
    # 4) Train loop
    # --------------------------------------------------------
    history = {"epoch": [], "train_loss": [], "dev_loss": [], "dev_corr": []}
    best_corr = -1.0

    print(f"Training for {num_epochs} epochs...")
    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device=device,
            scheduler=scheduler,
            grad_accum_steps=grad_accum_steps,
            amp=True,
        )

        dev_loss, dev_corr = eval_politeness(model, dev_loader, device=device)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_corr"].append(dev_corr)

        print(
            f"Epoch {epoch + 1}: "
            f"train_loss={tr_loss:.4f}  "
            f"dev_loss={dev_loss:.4f}  "
            f"dev_corr={dev_corr:.4f}"
        )

        if dev_corr > best_corr:
            best_corr = dev_corr
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "label2id": label2id,
                    "best_corr": best_corr,
                    "hparams": {
                        "architecture": "DirectionalSRL_Fusion",
                        "bert_name": bert_name,
                        "num_labels": len(label2id),
                        "use_indicator": True,
                        "use_distance": True,
                        "indicator_dim": 10,
                        "lstm_hidden": 768,
                        "mlp_hidden": 300,
                        "pos_dim": 50,
                        "max_distance": 128,
                        "dropout": 0.1,
                        "batch_size": 16,
                        "lr": lr,
                        "num_epochs": num_epochs,
                        "max_length": 128,
                    },
                },
                best_path,
            )
            print("new best correlation saved")

    # --------------------------------------------------------
    # 5) Save history
    # --------------------------------------------------------
    with open(hist_file_name, "wb") as f:
        pickle.dump(history, f)

    print(f"Best model saved to {best_path}")
    print(f"History saved to {hist_file_name}")

### Training - no fusion- mlp

In [ ]:
import os
import json
import pickle
from datetime import datetime
from typing import List, Dict

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup


# ============================================================
# Data processing
# ============================================================

def data_processing_for_loader(
    train_dev: List[UtteranceSample],
    train_sample: List[UtteranceSample],
    dev_sample: List[UtteranceSample],
    test_sample: List[UtteranceSample],
    tokenizer,
):
    """
    Build label2id from all frame labels in train+dev.
    Return utterance-level SRLDataset objects.
    """
    label2id: Dict[str, int] = {}

    for s in train_dev:
        for fr in s.frames:
            if fr is None or fr.labels is None:
                continue
            for l in fr.labels:
                if l not in label2id:
                    label2id[l] = len(label2id)

    id2label = {v: k for k, v in label2id.items()}

    train_ds = SRLDataset(train_sample, tokenizer, label2id, max_length=128, debug_print=False)
    dev_ds   = SRLDataset(dev_sample,   tokenizer, label2id, max_length=128, debug_print=False)
    test_ds  = SRLDataset(test_sample,  tokenizer, label2id, max_length=128, debug_print=False)

    return train_ds, dev_ds, test_ds, label2id, id2label


# ============================================================
# Main training
# ============================================================
if __name__ == "__main__":
    # --------------------------------------------------------
    # 0) Setup
    # --------------------------------------------------------
    bert_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    base_path = "/Enter-your-path/SRL_processed/"
    train_file = base_path + "srl_politeness_train.aligned.jsonl"
    dev_file   = base_path + "srl_politeness_dev.aligned.jsonl"
    test_file  = base_path + "test_update/srl_politeness_test.aligned.jsonl"

    data_class_train = create_utterance_samples(train_file)
    data_class_dev   = create_utterance_samples(dev_file)
    data_class_test  = create_utterance_samples(test_file)

    train_dev_data = data_class_train + data_class_dev

    train_ds, dev_ds, test_ds, label2id, id2label = data_processing_for_loader(
        train_dev_data,
        data_class_train,
        data_class_dev,
        data_class_test,
        tokenizer
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate = lambda b: srl_collate_ulevel(b, pad_token_id=pad_token_id, pad_label_id=-100)

    train_loader = DataLoader(
        train_ds,
        batch_size=16,
        shuffle=True,
        collate_fn=collate,
        num_workers=0,   # easier debugging
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=16,
        shuffle=False,
        collate_fn=collate,
        num_workers=0,
    )

    # --------------------------------------------------------
    # 1) Build model (fusion directional, mlp_hidden=300)
    # --------------------------------------------------------
    model = DirectionalSRL_nofusion_mlp(
        bert_name=bert_name,
        num_labels=len(label2id),
        use_indicator=True,
        use_distance=True,
        indicator_dim=10,
        lstm_hidden=768,
        mlp_hidden=mlp_layer,     # fixed as requested
        pos_dim=50,
        max_distance=128,
        dropout=0.1,
    ).to(device)

    # --------------------------------------------------------
    # 2) Optimizer / scheduler
    # --------------------------------------------------------
    num_epochs = 5
    grad_accum_steps = 1
    lr = 5e-5

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    total_steps = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # --------------------------------------------------------
    # 3) Save paths
    # --------------------------------------------------------
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")
    model_name = "directional_nofusion_mlp"

    save_dir = "/Enter-your-path/model_log"
    hist_dir = "/Enter-your-path/loss_history"
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(hist_dir, exist_ok=True)

    best_path = os.path.join(save_dir, today + "_best_model_"+model_name+"_epoch_{}.ckpt".format(str(num_epochs)))
    hist_file_name = os.path.join(hist_dir, today + "_model_"+model_name+"_epoch_{}.pkl".format(str(num_epochs)))

    # --------------------------------------------------------
    # 4) Train loop
    # --------------------------------------------------------
    history = {"epoch": [], "train_loss": [], "dev_loss": [], "dev_corr": []}
    best_corr = -1.0

    print(f"Training for {num_epochs} epochs...")
    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device=device,
            scheduler=scheduler,
            grad_accum_steps=grad_accum_steps,
            amp=True,
        )

        dev_loss, dev_corr = eval_politeness(model, dev_loader, device=device)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_corr"].append(dev_corr)

        print(
            f"Epoch {epoch + 1}: "
            f"train_loss={tr_loss:.4f}  "
            f"dev_loss={dev_loss:.4f}  "
            f"dev_corr={dev_corr:.4f}"
        )

        if dev_corr > best_corr:
            best_corr = dev_corr
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "label2id": label2id,
                    "best_corr": best_corr,
                    "hparams": {
                        "architecture": "DirectionalSRL_nofusion_mlp",
                        "bert_name": bert_name,
                        "num_labels": len(label2id),
                        "use_indicator": True,
                        "use_distance": True,
                        "indicator_dim": 10,
                        "lstm_hidden": 768,
                        "mlp_hidden": 300,
                        "pos_dim": 50,
                        "max_distance": 128,
                        "dropout": 0.1,
                        "batch_size": 16,
                        "lr": lr,
                        "num_epochs": num_epochs,
                        "max_length": 128,
                    },
                },
                best_path,
            )
            print("new best correlation saved")

    # --------------------------------------------------------
    # 5) Save history
    # --------------------------------------------------------
    with open(hist_file_name, "wb") as f:
        pickle.dump(history, f)

    print(f"Best model saved to {best_path}")
    print(f"History saved to {hist_file_name}")

### Training- no fusion - no mlp

In [ ]:
import os
import json
import pickle
from datetime import datetime
from typing import List, Dict

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup


# ============================================================
# Data processing
# ============================================================

def data_processing_for_loader(
    train_dev: List[UtteranceSample],
    train_sample: List[UtteranceSample],
    dev_sample: List[UtteranceSample],
    test_sample: List[UtteranceSample],
    tokenizer,
):
    """
    Build label2id from all frame labels in train+dev.
    Return utterance-level SRLDataset objects.
    """
    label2id: Dict[str, int] = {}

    for s in train_dev:
        for fr in s.frames:
            if fr is None or fr.labels is None:
                continue
            for l in fr.labels:
                if l not in label2id:
                    label2id[l] = len(label2id)

    id2label = {v: k for k, v in label2id.items()}

    train_ds = SRLDataset(train_sample, tokenizer, label2id, max_length=128, debug_print=False)
    dev_ds   = SRLDataset(dev_sample,   tokenizer, label2id, max_length=128, debug_print=False)
    test_ds  = SRLDataset(test_sample,  tokenizer, label2id, max_length=128, debug_print=False)

    return train_ds, dev_ds, test_ds, label2id, id2label


# ============================================================
# Main training
# ============================================================
if __name__ == "__main__":
    # --------------------------------------------------------
    # 0) Setup
    # --------------------------------------------------------
    bert_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    base_path = "/Enter-your-path/SRL_processed/"
    train_file = base_path + "srl_politeness_train.aligned.jsonl"
    dev_file   = base_path + "srl_politeness_dev.aligned.jsonl"
    test_file  = base_path + "test_update/srl_politeness_test.aligned.jsonl"

    data_class_train = create_utterance_samples(train_file)
    data_class_dev   = create_utterance_samples(dev_file)
    data_class_test  = create_utterance_samples(test_file)

    train_dev_data = data_class_train + data_class_dev

    train_ds, dev_ds, test_ds, label2id, id2label = data_processing_for_loader(
        train_dev_data,
        data_class_train,
        data_class_dev,
        data_class_test,
        tokenizer
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate = lambda b: srl_collate_ulevel(b, pad_token_id=pad_token_id, pad_label_id=-100)

    train_loader = DataLoader(
        train_ds,
        batch_size=16,
        shuffle=True,
        collate_fn=collate,
        num_workers=0,   # easier debugging
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=16,
        shuffle=False,
        collate_fn=collate,
        num_workers=0,
    )

    # --------------------------------------------------------
    # 1) Build model (fusion directional, mlp_hidden=256)
    # --------------------------------------------------------
    model = DirectionalSRL_nofusion_nomlp(
        bert_name=bert_name,
        num_labels=len(label2id),
        use_indicator=True,
        use_distance=True,
        indicator_dim=10,
        lstm_hidden=768,
        mlp_hidden=mlp_layer,     # fixed as requested
        pos_dim=50,
        max_distance=128,
        dropout=0.1,
    ).to(device)

    # --------------------------------------------------------
    # 2) Optimizer / scheduler
    # --------------------------------------------------------
    num_epochs = 5
    grad_accum_steps = 1
    lr = 5e-5

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    total_steps = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # --------------------------------------------------------
    # 3) Save paths
    # --------------------------------------------------------
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")

    save_dir = "/Enter-your-path/model_log"
    hist_dir = "/Enter-your-path/loss_history"
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(hist_dir, exist_ok=True)

    best_path = os.path.join(save_dir, today + "_best_model_4_directional_nofusion_no_mlp_epoch_{}.ckpt".format(str(num_epochs)))
    hist_file_name = os.path.join(hist_dir, today + "_model_directional_nofusion_no_mlp_epoch_{}.pkl".format(str(num_epochs)))

    # --------------------------------------------------------
    # 4) Train loop
    # --------------------------------------------------------
    history = {"epoch": [], "train_loss": [], "dev_loss": [], "dev_corr": []}
    best_corr = -1.0

    print(f"Training for {num_epochs} epochs...")
    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device=device,
            scheduler=scheduler,
            grad_accum_steps=grad_accum_steps,
            amp=True,
        )

        dev_loss, dev_corr = eval_politeness(model, dev_loader, device=device)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_corr"].append(dev_corr)

        print(
            f"Epoch {epoch + 1}: "
            f"train_loss={tr_loss:.4f}  "
            f"dev_loss={dev_loss:.4f}  "
            f"dev_corr={dev_corr:.4f}"
        )

        if dev_corr > best_corr:
            best_corr = dev_corr
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "label2id": label2id,
                    "best_corr": best_corr,
                    "hparams": {
                        "architecture": "DirectionalSRL_Fusion",
                        "bert_name": bert_name,
                        "num_labels": len(label2id),
                        "use_indicator": True,
                        "use_distance": True,
                        "indicator_dim": 10,
                        "lstm_hidden": 768,
                        "mlp_hidden": 300,
                        "pos_dim": 50,
                        "max_distance": 128,
                        "dropout": 0.1,
                        "batch_size": 16,
                        "lr": lr,
                        "num_epochs": num_epochs,
                        "max_length": 128,
                    },
                },
                best_path,
            )
            print("new best correlation saved")

    # --------------------------------------------------------
    # 5) Save history
    # --------------------------------------------------------
    with open(hist_file_name, "wb") as f:
        pickle.dump(history, f)

    print(f"Best model saved to {best_path}")
    print(f"History saved to {hist_file_name}")

# Vanila BERT model and Training (No SRL)

In [ ]:
class UtteranceOnlyDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=256):
        self.samples = samples  # List[UtteranceSample]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        utt = self.samples[idx]
        # Use a clean string: join tokens with space
        text = " ".join(utt.words)

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False,                 # collate will pad
            return_tensors=None
        )

        res = {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        }
        # Some tokenizers may not return token_type_ids (RoBERTa etc.)
        if "token_type_ids" in enc:
            res["token_type_ids"] = torch.tensor(enc["token_type_ids"], dtype=torch.long)

        if utt.politeness is not None:
            res["politeness"] = torch.tensor(float(utt.politeness), dtype=torch.float)

        return res


## Collate: pad to max --> creating tensor

In [ ]:
def utterance_collate(batch, pad_token_id: int):
    B = len(batch)
    max_L = max(x["input_ids"].size(0) for x in batch)

    input_ids = torch.full((B, max_L), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((B, max_L), dtype=torch.long)

    has_token_type = "token_type_ids" in batch[0]
    if has_token_type:
        token_type_ids = torch.zeros((B, max_L), dtype=torch.long)

    has_politeness = "politeness" in batch[0]
    if has_politeness:
        politeness = torch.zeros((B,), dtype=torch.float)

    for i, item in enumerate(batch):
        L = item["input_ids"].size(0)
        input_ids[i, :L] = item["input_ids"]
        attention_mask[i, :L] = item["attention_mask"]
        if has_token_type:
            token_type_ids[i, :L] = item["token_type_ids"]
        if has_politeness:
            politeness[i] = item["politeness"]

    res = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
    }
    if has_token_type:
        res["token_type_ids"] = token_type_ids
    if has_politeness:
        res["politeness"] = politeness

    return res


## Vanila BERT model

In [ ]:
# from transformers import AutoModel, AutoConfig

class VanillaBertPoliteness_mlp(nn.Module):
    def __init__(self, bert_name="bert-base-uncased", mlp_hidden=256, dropout=0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)
        H = self.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Sequential(
            nn.Linear(H, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1)
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None, politeness=None, **kwargs):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # Prefer pooler_output if available; else CLS token
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            pooled = out.pooler_output  # [B, H]
        else:
            pooled = out.last_hidden_state[:, 0, :]  # [B, H]

        pooled = self.dropout(pooled)

        # CHANGED: remove bounded regression
        # before:
        # score = 4.0 * torch.sigmoid(self.reg_head(pooled).squeeze(-1))
        # now:
        score = self.reg_head(pooled).squeeze(-1)  # [B]

        loss = torch.tensor(0.0, device=score.device)
        if politeness is not None:
            loss = nn.MSELoss()(score, politeness.float())

        return score, loss, None


from transformers import AutoModel, AutoConfig

#without additional reghead

class VanillaBertPoliteness(nn.Module):
    def __init__(self, bert_name="bert-base-uncased", dropout=0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)
        H = self.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Linear(H, 1)

    def forward(self, input_ids, attention_mask, token_type_ids=None, politeness=None, **kwargs):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            pooled = out.pooler_output
        else:
            pooled = out.last_hidden_state[:, 0, :]

        pooled = self.dropout(pooled)

        # unbounded regression output
        score = self.reg_head(pooled).squeeze(-1)

        loss = torch.tensor(0.0, device=score.device)
        if politeness is not None:
            loss = nn.MSELoss()(score, politeness.float())

        return score, loss, None

### Training - no MLP

In [ ]:
def data_processing_for_loader_vanilla(
    train_sample: List[UtteranceSample],
    dev_sample: List[UtteranceSample],
    test_sample: List[UtteranceSample],
    tokenizer,
    max_length: int = 256
):
    # **changed**: no label2id/id2label needed
    train_ds = UtteranceOnlyDataset(train_sample, tokenizer, max_length=max_length)
    dev_ds   = UtteranceOnlyDataset(dev_sample,   tokenizer, max_length=max_length)
    test_ds  = UtteranceOnlyDataset(test_sample,  tokenizer, max_length=max_length)

    # **changed**: return None placeholders so your main code won't break if it expects 5 outputs
    return train_ds, dev_ds, test_ds, None, None

if __name__ == '__main__': 
    
    
        # ===== 0. Setup Data and Init Model ======
    bert_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    

    
    base_path = "/Enter-your-path/SRL_processed/"
    data_class_train = create_utterance_samples(base_path + "srl_politeness_train.aligned.jsonl")
    data_class_dev = create_utterance_samples(base_path + "srl_politeness_dev.aligned.jsonl")
    data_class_test = create_utterance_samples(base_path + "test_update/srl_politeness_test.aligned.jsonl")
    
    # train_dev_data = data_class_train + data_class_dev 
    # train_bf_loader, dev_bf_loader,test_bf_loader, label2id, id2label = data_processing_for_loader_vanila(train_dev_data, data_class_train, data_class_dev, data_class_test, tokenizer)
    
    train_ds, dev_ds, test_ds, label2id, id2label = data_processing_for_loader_vanilla(
        data_class_train, data_class_dev, data_class_test, tokenizer, max_length=256
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate = lambda b: utterance_collate(b, pad_token_id=pad_token_id)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate)
    dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)
    test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, collate_fn=collate)


    # model = VanillaBertPoliteness(
    # bert_name=bert_name,
    # mlp_hidden=256,
    # dropout=0.1
    # ).to(device)
    
    model = VanillaBertPoliteness(
    bert_name=bert_name,
    # mlp_hidden=256,
    dropout=0.1
    ).to(device)
    
    
    # ===== 1. Init Optimizer =====
    num_epochs = 5
    grad_accum_steps = 1
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

    total_steps = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
      optimizer,
      num_warmup_steps=warmup_steps,
      num_training_steps=total_steps
    )

    # ===== 2. Start Training Loop =====
    history = {"epoch": [], "train_loss": [], "dev_loss": [], "dev_corr": []}
    best_corr = -1.0
    save_dir = "/Enter-your-path/model_log"
    
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")
    model_name = "vanilla_nomlp"
    
    # NOTE: Edit path based on intended model
    best_path = os.path.join(save_dir, "best_model_"+model_name+"_"+today+"_epoch_"+str(num_epochs)+".ckpt")
    
    print(f"Training for {num_epochs} epochs...")
    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(
            model, train_loader, optimizer, device=device,
            scheduler=scheduler, grad_accum_steps=grad_accum_steps, amp=True
        )
        
        # Evaluate politeness
        dev_loss, dev_corr = eval_politeness(model, dev_loader, device=device)
        
        # Log
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_corr"].append(dev_corr)

        print(f"Epoch {epoch+1}: train_loss={tr_loss:.4f}  dev_loss={dev_loss:.4f}  dev_corr={dev_corr:.4f}")
        
        # Save model if better correlation
        if dev_corr > best_corr:
            best_corr = dev_corr
            torch.save({
                "model_state": model.state_dict(),
                "label2id": label2id,
                "best_corr": best_corr,
                "hparams": {
                    "bert_name": bert_name,
                    # "num_labels": len(label2id),
                }
            }, best_path)
            print("new best correlation saved")
    
    # Save model history
    hist_path = "/Enter-your-path/loss_history/"
    # today = date.today()
    
    # NOTE: Edit path based on intended model
    hist_file_name = os.path.join(hist_path, today+"_"+model_name+"_epoch_"+str(num_epochs)+"_.pkl")
    
    with open(hist_file_name, 'wb') as f:
        pickle.dump(history, f)
    
    print(f"History saved to {hist_file_name}")
    print(f"Best model saved to {best_path}")
    

### Training -with MLP

In [ ]:
mlp_layer = 300

In [ ]:
def data_processing_for_loader_vanilla(
    train_sample: List[UtteranceSample],
    dev_sample: List[UtteranceSample],
    test_sample: List[UtteranceSample],
    tokenizer,
    max_length: int = 256
):
    # **changed**: no label2id/id2label needed
    train_ds = UtteranceOnlyDataset(train_sample, tokenizer, max_length=max_length)
    dev_ds   = UtteranceOnlyDataset(dev_sample,   tokenizer, max_length=max_length)
    test_ds  = UtteranceOnlyDataset(test_sample,  tokenizer, max_length=max_length)

    # **changed**: return None placeholders so your main code won't break if it expects 5 outputs
    return train_ds, dev_ds, test_ds, None, None

if __name__ == '__main__': 
    
    
        # ===== 0. Setup Data and Init Model ======
    bert_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    

    
    base_path = "/Enter-your-path/SRL_processed/"
    data_class_train = create_utterance_samples(base_path + "srl_politeness_train.aligned.jsonl")
    data_class_dev = create_utterance_samples(base_path + "srl_politeness_dev.aligned.jsonl")
    data_class_test = create_utterance_samples(base_path + "test_update/srl_politeness_test.aligned.jsonl")
    
    # train_dev_data = data_class_train + data_class_dev 
    # train_bf_loader, dev_bf_loader,test_bf_loader, label2id, id2label = data_processing_for_loader_vanila(train_dev_data, data_class_train, data_class_dev, data_class_test, tokenizer)
    
    train_ds, dev_ds, test_ds, label2id, id2label = data_processing_for_loader_vanilla(
        data_class_train, data_class_dev, data_class_test, tokenizer, max_length=256
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate = lambda b: utterance_collate(b, pad_token_id=pad_token_id)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate)
    dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)
    test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, collate_fn=collate)


    # model = VanillaBertPoliteness(
    # bert_name=bert_name,
    # mlp_hidden=256,
    # dropout=0.1
    # ).to(device)
    
    model = VanillaBertPoliteness_mlp(
    bert_name=bert_name,
    mlp_hidden=mlp_layer,
    dropout=0.1
    ).to(device)
    
    
    # ===== 1. Init Optimizer =====
    num_epochs = 5
    grad_accum_steps = 1
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

    total_steps = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
      optimizer,
      num_warmup_steps=warmup_steps,
      num_training_steps=total_steps
    )

    # ===== 2. Start Training Loop =====
    history = {"epoch": [], "train_loss": [], "dev_loss": [], "dev_corr": []}
    best_corr = -1.0
    save_dir = "/Enter-your-path/model_log"
    
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")
    model_name = "Vanilla_mlp"
    
    # NOTE: Edit path based on intended model
    best_path = os.path.join(save_dir, "best_"+model_name+"_"+today+"_epoch_"+str(num_epochs)+".ckpt")
    
    print(f"Training for {num_epochs} epochs...")
    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(
            model, train_loader, optimizer, device=device,
            scheduler=scheduler, grad_accum_steps=grad_accum_steps, amp=True
        )
        
        # Evaluate politeness
        dev_loss, dev_corr = eval_politeness(model, dev_loader, device=device)
        
        # Log
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_corr"].append(dev_corr)

        print(f"Epoch {epoch+1}: train_loss={tr_loss:.4f}  dev_loss={dev_loss:.4f}  dev_corr={dev_corr:.4f}")
        
        # Save model if better correlation
        if dev_corr > best_corr:
            best_corr = dev_corr
            torch.save({
                "model_state": model.state_dict(),
                "label2id": label2id,
                "best_corr": best_corr,
                "hparams": {
                    "bert_name": bert_name,
                    # "num_labels": len(label2id),
                }
            }, best_path)
            print("new best correlation saved")
    
    # Save model history
    hist_path = "/Enter-your-path/loss_history/"
    # today = date.today()
    
    # NOTE: Edit path based on intended model
    hist_file_name = os.path.join(hist_path, today+"_epoch_"+str(num_epochs)+"_"+model_name".pkl")
    
    with open(hist_file_name, 'wb') as f:
        pickle.dump(history, f)
    
    print(f"History saved to {hist_file_name}")
    

# Test (Utterance-level)

In [ ]:
# import json
# from typing import List
def create_utterance_samples(data_path: str) -> List[UtteranceSample]:
    samples = []
    with open(data_path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            samples.append(
                UtteranceSample(
                    words=ex["words"],
                    frames=[SRLFrame(**fr) for fr in ex.get("frames", [])],
                    politeness=ex.get("politeness", None),
                )
            )
    return samples

import torch

def load_model_and_labels(exp, device):
    print(f"Loading {exp['name']}...")
    ckpt_path = exp["ckpt_file"]
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)

    model_arch = exp.get("architecture", None)
    print(f'Loading {model_arch} ...')

    # For SRL models (token-label dependent)
    label2id = checkpoint.get("label2id", None)

    #       bert_name: str,
    #     num_labels: int,
    #     use_indicator: bool = True,
    #     indicator_dim: int = 10,
    #     lstm_hidden: int = 768,
    #     mlp_hidden: int = 300,
    #     dropout: float = 0.1,
    #     use_distance: bool = True,
    #     pos_dim: int = 50,
    #     max_distance: int = 128,
    #     num_roles: int = 6,
    #     role_dim: int = 32,
    #     attn_dim: int = 256,
    # ):
    
    
    if model_arch == "Directional_nofusion_mlp":
        if label2id is None:
            raise ValueError("DirectionalSRL_nofusion_mlp requires label2id in checkpoint.")
        model = DirectionalSRL_nofusion_mlp(
            bert_name="bert-base-uncased",
            num_labels=len(label2id),
            use_indicator=True, use_distance=True,
            indicator_dim=10, lstm_hidden=768,
            mlp_hidden=mlp_layer, pos_dim=50,
            max_distance=128, dropout=0.0
        )
    
    elif model_arch == "Directional_nofusion_nomlp":
        if label2id is None:
            raise ValueError("DirectionalSRL_nofusion_nomlp requires label2id in checkpoint.")
        model = DirectionalSRL_nofusion_nomlp(
            bert_name="bert-base-uncased",
            num_labels=len(label2id),
            use_indicator=True, use_distance=True,
            indicator_dim=10, lstm_hidden=768,
            mlp_hidden=mlp_layer, pos_dim=50,
            max_distance=128, dropout=0.0
        )
        
    elif model_arch == "Directional_fusion_mlp":
        if label2id is None:
            raise ValueError("DirectionalSRL_fusion_mlp requires label2id in checkpoint.")
        model = DirectionalSRL_fusion_mlp(
            bert_name="bert-base-uncased",
            num_labels=len(label2id),
            use_indicator=True, use_distance=True,
            indicator_dim=10, lstm_hidden=768,
            mlp_hidden=mlp_layer, pos_dim=50,
            max_distance=128, dropout=0.0
        )
        
    elif model_arch == "Directional_fusion_nomlp":
        if label2id is None:
            raise ValueError("DirectionalSRL_fusion_nomlp requires label2id in checkpoint.")
        model = DirectionalSRL_fusion_nomlp(
            bert_name="bert-base-uncased",
            num_labels=len(label2id),
            use_indicator=True, use_distance=True,
            indicator_dim=10, lstm_hidden=768,
            mlp_hidden=mlp_layer, pos_dim=50,
            max_distance=128, dropout=0.0
        )
        
    elif model_arch == "Vanilla_nomlp":
        model = VanillaBertPoliteness(
            bert_name="bert-base-uncased",
            # mlp_hidden=256,
            dropout=0.0
        )
        label2id = None  # not used
        
    elif model_arch == "Vanilla_mlp":
        model = VanillaBertPoliteness_mlp(
            bert_name="bert-base-uncased",
            mlp_hidden=mlp_layer,
            dropout=0.0
        )
        label2id = None  # not used
    else:
        raise ValueError(f"Unknown architecture: {model_arch}")

    model.load_state_dict(checkpoint["model_state"], strict=False)
    model.to(device)
    model.eval()
    return model, label2id




In [ ]:
def get_dataloader_srl_aligned(data_file, tokenizer, label2id, batch_size=32):
    if label2id is None:
        raise ValueError("SRL dataloader requires label2id.")

    data_samples = create_utterance_samples(data_file)

    # IMPORTANT: this SRLDataset must be your UPDATED multi-frame dataset class
    ds = SRLDataset(data_samples, tokenizer, label2id, max_length=256)

    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate_fn = lambda b: srl_collate_ulevel(b, pad_token_id=pad_id, pad_label_id=-100)

    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    return dl, data_samples

def get_dataloader_vanilla(data_file, tokenizer, batch_size=32):
    data_samples = create_utterance_samples(data_file)

    ds = UtteranceOnlyDataset(data_samples, tokenizer, max_length=256)

    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate_fn = lambda b: utterance_collate(b, pad_token_id=pad_id)

    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    return dl, data_samples


In [ ]:
import numpy as np
from scipy.stats import pearsonr, spearmanr

@torch.no_grad()
def test_politeness(model, dataloader, device="cuda"):
    model.eval()
    all_preds, all_golds = [], []
    total_loss, n_batches = 0.0, 0

    for batch in dataloader:
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

        preds, loss, _ = model(**batch)
        total_loss += float(loss.item())
        n_batches += 1

        if "politeness" in batch:
            all_preds.append(preds.detach().cpu().numpy())
            all_golds.append(batch["politeness"].detach().cpu().numpy())

    avg_loss = total_loss / max(1, n_batches)

    if len(all_preds) == 0:
        return avg_loss, 0.0, None, None

    preds = np.concatenate(all_preds, axis=0)
    golds = np.concatenate(all_golds, axis=0)

    # if len(preds) > 1:
    #     corr, _ = pearsonr(golds, preds)
    # else:
    #     corr = 0.0
    
    if len(preds) > 1:
        pearson_corr, pearson_p = pearsonr(golds, preds)
        spearman_corr, spearman_p = spearmanr(golds, preds)
    else:
        pearson_corr, pearson_p = 0.0, 1.0
        spearman_corr, spearman_p = 0.0, 1.0


    # return avg_loss, corr, preds, golds
    return  avg_loss, pearson_corr, pearson_p, spearman_corr, spearman_p, preds, golds


def run_test(exp, test_file, device="cuda", batch_size=32, save_csv_path=None):
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

    model, label2id = load_model_and_labels(exp, device)

    if exp["architecture"] == "Vanilla":
        dl, samples = get_dataloader_vanilla(test_file, tokenizer, batch_size=batch_size)
    elif exp["architecture"] == "Vanilla_mlp":
        dl, samples = get_dataloader_vanilla(test_file, tokenizer, batch_size=batch_size)
    # Vanilla_nomlp
    elif exp["architecture"] == "Vanilla_nomlp":
        dl, samples = get_dataloader_vanilla(test_file, tokenizer, batch_size=batch_size)
    
    elif exp["architecture"] == "DirectionalSRL_nofusion_mlp":
        dl, samples = get_dataloader_srl_aligned(test_file, tokenizer, label2id, batch_size=batch_size)
    
    elif exp["architecture"] == "DirectionalSRL_fusion_mlp":
        dl, samples = get_dataloader_srl_aligned(test_file, tokenizer, label2id, batch_size=batch_size)
        
    elif exp["architecture"] == "DirectionalSRL_nofusion_nomlp":
        dl, samples = get_dataloader_srl_aligned(test_file, tokenizer, label2id, batch_size=batch_size)
    
    elif exp["architecture"] == "DirectionalSRL_fusion_nomlp":
        dl, samples = get_dataloader_srl_aligned(test_file, tokenizer, label2id, batch_size=batch_size)
    
    else:
        dl, samples = get_dataloader_srl_aligned(test_file, tokenizer, label2id, batch_size=batch_size)

    avg_loss, pearson_corr, pearson_p, spearman_corr, spearman_p, preds, golds = test_politeness(model, dl, device=device)
    print(
                f"[{exp['name']}] "
                f"test_loss={avg_loss:.4f}  "
                f"test_pearson_corr={pearson_corr:.4f} "
                f"p={pearson_p:.3e}  "
                f"test_spearman_corr={spearman_corr:.4f} "
                f"p={spearman_p:.3e}"
            )

    if save_csv_path is not None and preds is not None:
        rows = []
        for i, s in enumerate(samples):
            rows.append({
                "idx": i,
                "text": " ".join(s.words),
                "gold_politeness": float(s.politeness) if s.politeness is not None else None,
                "pred_politeness": float(preds[i]),
            })
        pd.DataFrame(rows).to_csv(save_csv_path, index=False)
        print(f"Saved predictions to: {save_csv_path}")

    return avg_loss, pearson_corr, spearman_corr


In [ ]:
if __name__ == '__main__':
    device = "cuda" if torch.cuda.is_available() else "cpu"
    test_file = "/Enter-your-path/srl_politeness_test.aligned.jsonl"

    exp = {
        "name": "Vanila_BERT_nomlp",
        "architecture": "Vanilla_nomlp",
        "ckpt_file": "/Enter-your-path/best_model_vanilla_nomlp_2026-04-26_15-29-04_epoch_5.ckpt",
    }
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")
    model_name = "vanilla_nomlp"
    
    run_test(exp, test_file, device=device, batch_size=16, save_csv_path="/Enter-your-path/model_log/"+today+"_"+model_name+".csv")


In [ ]:
if __name__ == '__main__':
    device = "cuda" if torch.cuda.is_available() else "cpu"
    test_file = "/Enter-your-path/srl_politeness_test.aligned.jsonl"

    exp = {
        "name": "Vanila_BERT_withmlp",
        "architecture": "Vanilla_mlp",
        "ckpt_file": "/Enter-your-path/best_model_vanila_BERT_mlp_2026-04-24_20-24-21_epoch_5.ckpt",
    }
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")
    
    run_test(exp, test_file, device=device, batch_size=16, save_csv_path="/Enter-your-path/vanila_BERT_test_preds_2_mlp_"+today+".csv")


In [ ]:
import numpy as np

if __name__ == '__main__':
    device = "cuda" if torch.cuda.is_available() else "cpu"
    test_file = "/Enter-your-path/srl_politeness_test.aligned.jsonl"

    exp = {
        "name": "Directional_SRL",
        "architecture": "Directional_fusion_nomlp",
        "ckpt_file": "/Enter-your-path/2026-04-24_19-09-40_best_model_4_directional_fusion_no_mlp_epoch_5.ckpt",
    }
    
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")

    run_test(exp, test_file, device=device, batch_size=16, save_csv_path="/Enter-your-path/directional_test_preds_fusion_"+today+".csv")


In [ ]:
import numpy as np

if __name__ == '__main__':
    device = "cuda" if torch.cuda.is_available() else "cpu"
    test_file = "/Enter-your-path/srl_politeness_test.aligned.jsonl"

    exp = {
        "name": "Directional_SRL_nofusion",
        "architecture": "Directional_nofusion_nomlp",
        "ckpt_file": "/Enter-your-path/2026-04-24_20-13-19_best_model_4_directional_nofusion_no_mlp_epoch_5.ckpt",
    }
    
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")

    run_test(exp, test_file, device=device, batch_size=16, save_csv_path="/Enter-your-path/directional_test_preds_nofusion_"+today+".csv")


In [ ]:
import numpy as np

if __name__ == '__main__':
    device = "cuda" if torch.cuda.is_available() else "cpu"
    test_file = "/Enter-your-path/srl_politeness_test.aligned.jsonl"

    exp = {
        "name": "Directional_SRL_nofusion_mlp",
        "architecture": "Directional_nofusion_mlp",
        "ckpt_file": "/Enter-your-path/2026-04-26_17-23-22_best_model_directional_nofusion_mlp_epoch_5.ckpt",
    }
    
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")
    model_name = "directional_nofusion_mlp"

    run_test(exp, test_file, device=device, batch_size=16, save_csv_path="/Enter-your-path/model_log/"+model_name+"_"+today+".csv")


In [ ]:
import numpy as np

if __name__ == '__main__':
    device = "cuda" if torch.cuda.is_available() else "cpu"
    test_file = "/Enter-your-path/srl_politeness_test.aligned.jsonl"

    exp = {
        "name": "Directional_SRL_fusion_mlp",
        "architecture": "Directional_fusion_mlp",
        "ckpt_file": "/Enter-your-path/2026-04-26_17-22-34_best_model_directional_fusion_mlp_epoch_5.ckpt",
    }
    
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")
    model_name = "directional_fusion_mlp"

    run_test(exp, test_file, device=device, batch_size=16, save_csv_path="/Enter-your-path/model_log/"+model_name+"_"+today+".csv")
